# README

- **Author**: 詹明翰、謝子詮、曾翊涵（共同作者沿用學生 Unit 1.1 明確指示，不推定個別分工）
- **Created At**: 2026-11-26
- **Last Modified At**: 2026-11-26

## What does this file do?

跨年度交通事故資料正式驗證及候選清理規則設計。2024 選第一當事人、2025 保留現有第一當事人，
逐年獨立診斷；只在記憶體測試規則及特徵有效分母，不合併兩年的事故列。
候選事件鍵不是官方 ID；45,130 筆不是完成空間配對後的最終事故數。
規則分為學生已確認的分析口徑、官方表單支持的定義、仍待確認的建議規則。

依教師 `templates/data/code/template-data-processing-code.ipynb` 保留本 README 與 input/output 欄位順序。
本單元明確不授權正式處理，因此將範本的轉換／寫檔區替換為唯讀診斷，不產出 processed 檔案。

## What does this file take?

- **Source Data Sets / supporting documentation**（精確逐檔，不使用 glob）：
  - `data/raw/113年-臺北市死傷交通事故明細.csv`；SHA-256：`b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5`
  - `data/raw/114年-臺北市死傷交通事故明細.csv`；SHA-256：`61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c`
  - `data/raw/事故對照表.pdf`；SHA-256：`f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670`
  - `data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb`；SHA-256：`0107387bd2362bd4c1a4db1056e5dbddcaad88cc362435d1236c1d7a352f4381`
- CSV 編碼 UTF-8-SIG；原始來源為[臺北市死傷交通事故資料](https://data.taipei/dataset/detail?id=2f238b4f-1b27-4085-93e9-d684ef0e2735)。
- 原始 CSV／PDF hash 沿用 Unit 0.5 綁定；上游 Notebook hash 為本次讀取版本。先檢查再讀取，版本不同即停止。
- 原始下載日期／本機與官方當前資源的位元組對應仍未知；不以本次驗證日期代填。

## What does this file output?

- 本 Notebook 保存：分析口徑與 ID 設計、49 列完整跨年 schema、官方碼表與逐年分布、
  全欄缺失／特殊值／碼外值、座標來源證據、Dataset A／C 候選定義、規則影響與非巢狀 Sample Flow Table。
- Output key: `validation_documentation`；Role: 驗證文件，**不是 analysis／train／test 資料檔**。
- Expected schema and row-count rules: 診斷表維度與各表所列分母；保留與 Unit 0.5 的實際比對結果，不強迫數據符合舊值。
- Verification status / SHA-256: 實際狀態見各 code cell 保存輸出及末尾輸入保護檢查；文件不是原始資料，沒有 cleaned-data hash。
- Downstream finding input: not applicable；未建立清理後事故表或 Dataset A／C，不能把本文件的診斷表當正式 finding 輸入。
- 不修改原始／上游文件，不讀取人口或都市計畫圖資，不指定未知 CRS、不作空間歸里、分群、預測或統計推論。

所有程式使用 `.venv/Scripts/python.exe`。相對路徑從最近含 AGENTS.md 與 templates/ 的工作包根目錄定位。


In [1]:
from pathlib import Path
import ast, csv, hashlib, json, os, platform, sys, importlib.metadata as metadata
import numpy as np
import pandas as pd
import nbformat
import pymupdf
from IPython.display import display, Markdown

def find_project_root(start=None):
    start=Path.cwd() if start is None else Path(start)
    for p in (start.resolve(),*start.resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within the project containing AGENTS.md and templates/.')

PROJECT_ROOT=find_project_root()
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
assert sys.prefix!=sys.base_prefix
os.environ['PROJ_NETWORK']='OFF'
pd.set_option('display.max_colwidth',110)

def project_file(relative,area='raw'):
    rel=Path(relative)
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Project-relative exact paths required')
    p=(PROJECT_ROOT/rel).resolve()
    if not p.is_relative_to((PROJECT_ROOT/'data'/area).resolve()):raise ValueError('Input outside allowed area')
    if not p.is_file():raise FileNotFoundError(p)
    return p

def sha256_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def show(title,frame,block=20):
    display(Markdown('**'+title+'**'))
    if not len(frame):display(Markdown('無符合紀錄。'))
    for start in range(0,len(frame),block):display(frame.iloc[start:start+block])

def numeric(s):return pd.to_numeric(s,errors='coerce')

def nonnegative_integer(s):
    x=numeric(s)
    return (x.notna() & np.isfinite(x) & x.ge(0) & x.mod(1).eq(0)).fillna(False)

print({'python_version':platform.python_version(),'interpreter_matches_project_venv':True,
       'package_versions':{k:metadata.version(k) for k in ['pandas','numpy','nbformat','nbclient','pymupdf','pyproj']},
       'network_in_notebook':'none; source evidence was inspected separately on 2026-11-26'})


{'python_version': '3.12.2', 'interpreter_matches_project_venv': True, 'package_versions': {'pandas': '3.1.1', 'numpy': '2.5.5', 'nbformat': '5.11.1', 'nbclient': '0.11.0', 'pymupdf': '2.1', 'pyproj': '3.8.0'}, 'network_in_notebook': 'none; source evidence was inspected separately on 2026-11-26'}


In [2]:
EXPECTED_SHA256 = {'data/raw/113年-臺北市死傷交通事故明細.csv': 'b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5', 'data/raw/114年-臺北市死傷交通事故明細.csv': '61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c', 'data/raw/事故對照表.pdf': 'f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670', 'data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb': '0107387bd2362bd4c1a4db1056e5dbddcaad88cc362435d1236c1d7a352f4381'}
INPUTS={2024:'data/raw/113年-臺北市死傷交通事故明細.csv',2025:'data/raw/114年-臺北市死傷交通事故明細.csv'}
OUTPUTS={}  # This unit explicitly creates no processed datasets.
manifest=[]
for relative,expected in EXPECTED_SHA256.items():
    p=project_file(relative)
    actual=sha256_file(p)
    if actual!=expected:raise ValueError(f'Input differs from bound version: {relative}')
    manifest.append({'input':relative,'bytes':p.stat().st_size,'SHA-256':actual})
show('表 1：固定輸入與逐檔身分檢查',pd.DataFrame(manifest))
upstream=nbformat.read(project_file('data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb'),as_version=4)
upstream_binding=next(c.source for c in upstream.cells if c.cell_type=='code' and c.source.startswith('EXPECTED_SHA256 ='))
unit05_hashes=ast.literal_eval(ast.parse(upstream_binding).body[0].value)
assert all(EXPECTED_SHA256[p]==digest for p,digest in unit05_hashes.items())
raw={year:pd.read_csv(project_file(path),encoding='utf-8-sig',low_memory=False) for year,path in INPUTS.items()}
with pymupdf.open(project_file('data/raw/事故對照表.pdf')) as document:
    pdf_text=[page.get_text() for page in document]
assert len(pdf_text)==4 and '112.07.01' in pdf_text[0]
print({'Unit_0_5_raw_bindings_match':True,'PDF_pages':len(pdf_text),'PDF_effective_label':'112.07.01 後適用'})


**表 1：固定輸入與逐檔身分檢查**

,input,bytes,SHA-256
0,data/raw/113年-臺北市死傷交通事故明細.csv,9398043,b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5
1,data/raw/114年-臺北市死傷交通事故明細.csv,4082763,61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c
2,data/raw/事故對照表.pdf,487979,f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670
3,data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb,338980,0107387bd2362bd4c1a4db1056e5dbddcaad88cc362435d1236c1d7a352f4381


{'Unit_0_5_raw_bindings_match': True, 'PDF_pages': 4, 'PDF_effective_label': '112.07.01 後適用'}


## 1. Analytical Unit and Record Identity
學生已確認採第一當事人作候選事件；其他當事人原始列完整保留。候選鍵採年月日時分＋地點＋X／Y，
groupby 明確設 dropna=False，避免遺失缺座標紀錄。群內一致只支持操作口徑，不能認證官方事件身分。

研究用 ID 建議使用 `tpe-crash:{source_year}:{原始檔完整SHA256}:r{source_row_number:06d}`。
source_row_number 是排除表頭後、CSV parser 的 1-based 邏輯資料列序，不是實體文字行號；
保留篩選前序次、不因第一當事人篩選重編號。hash改變即不同輸入版本，不默默延用 ID。
這只是本次記憶體可重現性驗證，不輸出事件表；record_id 不是官方事故 ID，也不由有缺失的候選鍵拼成。


In [3]:
KEY=['發生年度','發生月','發生日','發生時-Hours','發生分','肇事地點','座標-X','座標-Y']
candidates={year:frame.loc[frame['當事人序號'].eq(1)] for year,frame in raw.items()}
unit_rows=[]
lineage_examples=[]
record_id_sets={}
for year,frame in raw.items():
    grouped=frame.groupby(KEY,dropna=False,sort=False)
    scene=[c for c in frame.columns[:30] if c not in ['車種','當事人序號']]
    counts=grouped['當事人序號'].agg(lambda s:int(s.eq(1).sum()))
    consistency=grouped[scene].nunique(dropna=False).gt(1).sum().rename('有不一致的候選群組數').to_frame()
    show(f'表 2：{year} 所有候選事故情境／總量欄群內一致性；缺失亦列為狀態',consistency)
    first=candidates[year]
    unit_rows.append({'year':year,'原始當事人列':len(frame),'原始欄數':len(frame.columns),
       '原始完整列重複':int(frame.duplicated().sum()),'候選事件鍵群':grouped.ngroups,
       'groupby保留列數':int(grouped.size().sum()),'群內第一當事人非恰一筆':int(counts.ne(1).sum()),
       '第一當事人候選':len(first),'第一當事人候選鍵重複':int(first.duplicated(KEY).sum())})
    with project_file(INPUTS[year]).open('r',encoding='utf-8-sig',newline='') as f:
        parsed=list(csv.reader(f))
    assert len(parsed)-1==len(frame) and all(parsed[1:]),'Review blank records before assigning row numbers'
    lineage=pd.DataFrame({'source_year':year,'source_file':INPUTS[year],
                          'source_row_number':first.index.to_numpy()+1})
    lineage['record_id']=[f'tpe-crash:{year}:{EXPECTED_SHA256[INPUTS[year]]}:r{n:06d}' for n in lineage.source_row_number]
    assert lineage.record_id.is_unique
    record_id_sets[year]=set(lineage.record_id)
    show(f'表 3：{year} 研究用 ID 前三筆示例；非正式事故表',lineage.head(3))
assert not (record_id_sets[2024]&record_id_sets[2025])
show('表 4：候選事件單位與重複診斷',pd.DataFrame(unit_rows))
comparison=[]
for year,previous in [(2024,22368),(2025,22762)]:
    comparison.append({'year':year,'Unit 0.5 候選數':previous,'本次觀測':len(candidates[year]),'一致':previous==len(candidates[year])})
show('表 5：與 Unit 0.5 對照；若不同據實標示，不強制修正',pd.DataFrame(comparison))
print({'candidate_total':sum(len(f) for f in candidates.values()),'not_final_GIS_sample':True,
       'record_ids_unique_within_and_between_years':True,'official_accident_ID':'not available'})


**表 2：2024 所有候選事故情境／總量欄群內一致性；缺失亦列為狀態**

,有不一致的候選群組數
發生年度,0
發生月,0
發生日,0
發生時-Hours,0
發生分,0
區序,0
肇事地點,0
死亡人數,0
2-30日死亡人數,0
受傷人數,0


,有不一致的候選群組數
道路障礙2,0
號誌1,0
號誌2,0
車道劃分-分向,0
車道劃分-分道1,0
車道劃分-分道2,0
車道劃分-分道3,0
事故類型及型態,0


**表 3：2024 研究用 ID 前三筆示例；非正式事故表**

,source_year,source_file,source_row_number,record_id
0,2024,data/raw/113年-臺北市死傷交通事故明細.csv,1,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000001
1,2024,data/raw/113年-臺北市死傷交通事故明細.csv,3,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000003
2,2024,data/raw/113年-臺北市死傷交通事故明細.csv,6,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r000006


**表 2：2025 所有候選事故情境／總量欄群內一致性；缺失亦列為狀態**

... (5 outputs left out)

,year,原始當事人列,原始欄數,原始完整列重複,候選事件鍵群,groupby保留列數,群內第一當事人非恰一筆,第一當事人候選,第一當事人候選鍵重複
0,2024,51810,49,0,22368,51810,0,22368,0
1,2025,22762,47,0,22762,22762,0,22762,0


**表 5：與 Unit 0.5 對照；若不同據實標示，不強制修正**

,year,Unit 0.5 候選數,本次觀測,一致
0,2024,22368,22368,True
1,2025,22762,22762,True


{'candidate_total': 45130, 'not_final_GIS_sample': True, 'record_ids_unique_within_and_between_years': True, 'official_accident_ID': 'not available'}


## 2. Cross-Year Schema Harmonization
完整對應表 49 列覆蓋 2024 的 49 個原始欄及 2025 的 47 個原始欄；
道路照明設備／光線放在同一「候選」對應列，不表示已核准合併。
A＝原值可對齊；B＝須曆法／型態轉換；C＝定義／代碼語意仍待確認；D＝年度專屬。
E 是研究關聯維度，另列「不預計主分析」，可與 A–D 並存，不因此遺失可比較性資訊。
英文名稱、型態及處理均是後續建議；本 Notebook 不實施正式 schema 合併或產出清理表。


In [4]:
SCHEMA = [{'field_id': 'F01', '2024原始欄': '發生年度', '2025原始欄': '發生年度', '建議英文名': 'accident_year', '中文定義': '事故發生年，檔內為民國 113／114；對應西元時在記憶體加 1911', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故時間', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'B', '可比較性': '可透過明確曆法／時分轉換對齊；不於本單元輸出轉換表', '建議處理': '先保留民國年／日期時間原欄，另驗證曆法及時分；西元年=民國年+1911', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF p.1 發生時間；原始值與檔名核對', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F02', '2024原始欄': '發生月', '2025原始欄': '發生月', '建議英文名': 'accident_month', '中文定義': '事故發生月份', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故時間', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'B', '可比較性': '可透過明確曆法／時分轉換對齊；不於本單元輸出轉換表', '建議處理': '先保留民國年／日期時間原欄，另驗證曆法及時分；西元年=民國年+1911', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF p.1 發生時間', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F03', '2024原始欄': '發生日', '2025原始欄': '發生日', '建議英文名': 'accident_day', '中文定義': '事故發生日', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故時間', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'B', '可比較性': '可透過明確曆法／時分轉換對齊；不於本單元輸出轉換表', '建議處理': '先保留民國年／日期時間原欄，另驗證曆法及時分；西元年=民國年+1911', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF p.1 發生時間', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F04', '2024原始欄': '發生時-Hours', '2025原始欄': '發生時-Hours', '建議英文名': 'accident_hour', '中文定義': '事故發生小時；檔內 24 小時制數值', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故時間', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'B', '可比較性': '可透過明確曆法／時分轉換對齊；不於本單元輸出轉換表', '建議處理': '先保留民國年／日期時間原欄，另驗證曆法及時分；西元年=民國年+1911', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF p.1 發生時間；檔內有效性檢查', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F05', '2024原始欄': '發生分', '2025原始欄': '發生分', '建議英文名': 'accident_minute', '中文定義': '事故發生分鐘', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故時間', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'B', '可比較性': '可透過明確曆法／時分轉換對齊；不於本單元輸出轉換表', '建議處理': '先保留民國年／日期時間原欄，另驗證曆法及時分；西元年=民國年+1911', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF p.1 發生時間', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F06', '2024原始欄': '區序', '2025原始欄': '區序', '建議英文名': 'district_label_source', '中文定義': '本機字串含行政區序號及區名；正式區序代碼規格未知', '建議型態': 'string（保留原始碼／文字）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '身分／未來空間驗證輔助', '定義依據及未知': '官方頁列欄名；檔內觀測，不自行當里代碼', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F07', '2024原始欄': '肇事地點', '2025原始欄': '肇事地點', '建議英文名': 'accident_location_text', '中文定義': '事故地點文字；不是當事人居住地址', '建議型態': 'string（保留原始碼／文字）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '身分／未來空間驗證輔助', '定義依據及未知': 'PDF p.1 發生地點；發布 CSV 轉錄細節未知', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F08', '2024原始欄': '死亡人數', '2025原始欄': '死亡人數', '建議英文名': 'fatalities_24h_count_candidate', '中文定義': '事故死傷人數中的 24 小時內死亡人數；不含個人受傷程度代碼', '建議型態': 'Int64（先查非負整數；原值另保留）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '人數（非負整數候選）；缺失不等於0', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': 'C剖析候選', '定義依據及未知': 'PDF p.1 死傷人數；與官方頁 24 小時定義對照', '是否預計主要分析': 'C剖析候選；死亡口徑待補'}, {'field_id': 'F09', '2024原始欄': '2-30日死亡人數', '2025原始欄': '2-30日死亡人數', '建議英文名': 'fatalities_2_30d_count_candidate', '中文定義': '欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認', '建議型態': 'Int64（先查非負整數；原值另保留）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '人數（非負整數候選）；缺失不等於0', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': 'C剖析候選', '定義依據及未知': 'CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式', '是否預計主要分析': 'C剖析候選；死亡口徑待補'}, {'field_id': 'F10', '2024原始欄': '受傷人數', '2025原始欄': '受傷人數', '建議英文名': 'injured_count', '中文定義': '事故受傷人數；與單一當事人的受傷程度不同', '建議型態': 'Int64（先查非負整數；原值另保留）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '人數（非負整數候選）；缺失不等於0', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'A候選特徵輸入；亦為C剖析', '定義依據及未知': 'PDF p.1 死傷人數', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F11', '2024原始欄': '當事人序號', '2025原始欄': '當事人序號', '建議英文名': 'party_sequence', '中文定義': '事故內當事人序次的候選欄；1 為第一當事人紀錄', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': '未取得完整發布規格', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '身分／未來空間驗證輔助', '定義依據及未知': 'PDF p.3 第幾當事者；不是事故官方 ID，也不等同肇責認定', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F12', '2024原始欄': '車種', '2025原始欄': '車種', '建議英文名': 'first_party_vehicle_code', '中文定義': '當事者區分類別，包括車輛及人', '建議型態': 'string（保留原始碼／文字）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉖；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '補充變數', '定義依據及未知': 'PDF p.3 類別表；如 C03 普通重型機車、H01 行人', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F13', '2024原始欄': '天候', '2025原始欄': '天候', '建議英文名': 'weather_code', '中文定義': '事故天候代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ④；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '補充變數', '定義依據及未知': 'PDF p.1 天候；例 6 雨、7 陰、8 晴', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F14', '2024原始欄': '道路照明設備', '2025原始欄': '光線', '建議英文名': 'road_lighting_code_candidate', '中文定義': '2024 欄名；與 PDF 照明欄候選對應', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑤；碼集見表8', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F15', '2024原始欄': '道路類別', '2025原始欄': '道路類別', '建議英文名': 'road_class_first_party_code', '中文定義': '事故道路類別代碼，表單註明第 1 當事者', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事者所在道路；PDF⑥', '官方碼或單位': 'PDF p.1 ⑥；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1；例 5 市區道路', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F16', '2024原始欄': '速限-速度限制', '2025原始欄': '速限-速度限制', '建議英文名': 'speed_limit_source', '中文定義': '道路速限，不是當事人實際車速；0、502 等值不自行更正', '建議型態': 'Float64（nullable；原始數值另保留）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '道路速限；CSV單位規格待補，km/h為候選', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': '補充變數', '定義依據及未知': 'PDF p.1 道路速限；CSV 單位及 0 的發布規則待官方確認', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F17', '2024原始欄': '道路型態', '2025原始欄': '道路型態', '建議英文名': 'road_shape_code', '中文定義': '事故道路型態代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑧；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '補充變數', '定義依據及未知': 'PDF p.1；例 03 三岔、04 四岔、14 直路', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F18', '2024原始欄': '事故位置', '2025原始欄': '事故位置', '建議英文名': 'accident_position_code', '中文定義': '事故發生位置類別', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑨；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F19', '2024原始欄': '路面狀況1', '2025原始欄': '路面狀況1', '建議英文名': 'surface_material_code', '中文定義': '路面鋪裝代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑩(1)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 路面狀況 (1)', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F20', '2024原始欄': '路面狀況2', '2025原始欄': '路面狀況2', '建議英文名': 'surface_condition_code', '中文定義': '路面狀態代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑩(2)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 路面狀況 (2)；5 乾燥', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F21', '2024原始欄': '路面狀況3', '2025原始欄': '路面狀況3', '建議英文名': 'surface_defect_code', '中文定義': '路面缺陷代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑩(3)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 路面狀況 (3)；4 無缺陷', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F22', '2024原始欄': '道路障礙1', '2025原始欄': '道路障礙1', '建議英文名': 'road_obstacle_code', '中文定義': '障礙物代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑪(1)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 道路障礙 (1)；5 無障礙物', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F23', '2024原始欄': '道路障礙2', '2025原始欄': '道路障礙2', '建議英文名': 'sight_condition_code', '中文定義': '視距代碼；新版及表單內舊碼不可混用', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑪(2)；碼集見表8', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 道路障礙 (2) 同時出現版本相關項目；13 良好', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F24', '2024原始欄': '號誌1', '2025原始欄': '號誌1', '建議英文名': 'signal_type_code', '中文定義': '號誌種類代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑫(1)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 號誌 (1)；4 無號誌', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F25', '2024原始欄': '號誌2', '2025原始欄': '號誌2', '建議英文名': 'signal_status_code', '中文定義': '號誌動作代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑫(2)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 號誌 (2)；1 正常、4 無號誌', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F26', '2024原始欄': '車道劃分-分向', '2025原始欄': '車道劃分-分向', '建議英文名': 'direction_divider_code', '中文定義': '分向設施代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑬；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 分向設施；10 無分向設施', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F27', '2024原始欄': '車道劃分-分道1', '2025原始欄': '車道劃分-分道1', '建議英文名': 'general_lane_divider_code', '中文定義': '快車道或一般車道間分道設施代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑭(1)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 分道設施 (1)', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F28', '2024原始欄': '車道劃分-分道2', '2025原始欄': '車道劃分-分道2', '建議英文名': 'fast_slow_lane_divider_code', '中文定義': '快慢車道間分道設施代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑭(2)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 分道設施 (2)', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F29', '2024原始欄': '車道劃分-分道3', '2025原始欄': '車道劃分-分道3', '建議英文名': 'road_edge_line_code', '中文定義': '路面邊線代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑭(3)；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.1 分道設施 (3)；1 有、2 無', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F30', '2024原始欄': '事故類型及型態', '2025原始欄': '事故類型及型態', '建議英文名': 'accident_type_code', '中文定義': '事故情境分類，不等同所有涉入者的完整屬性', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': 'PDF p.1 ⑮；車與車定義另見p.2；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'A候選特徵輸入', '定義依據及未知': 'PDF pp.1–2；1–9 人與車、13 追撞、16 側撞', '是否預計主要分析': 'A候選輸入；尚非最終特徵'}, {'field_id': 'F31', '2024原始欄': '性別', '2025原始欄': '性別', '建議英文名': 'first_party_sex_code', '中文定義': '當事者屬性別代碼，含非自然人及未查獲等特殊類別', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ⑰；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '補充變數', '定義依據及未知': 'PDF p.3；1 男、2 女、3 無或物、4 肇逃未查獲、5 無人駕駛', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F32', '2024原始欄': '年齡', '2025原始欄': '年齡', '建議英文名': 'first_party_age_source', '中文定義': '當事者年齡數值；年齡基準、-1／0 等特殊值定義未知', '建議型態': 'Float64（nullable；原始數值另保留）', '觀察層級': '第一當事人層級', '官方碼或單位': '歲；特殊值及計齡基準未知', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': '補充變數', '定義依據及未知': 'PDF p.3 原表為出生年月日；CSV 年齡轉換及特殊值需補證', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F33', '2024原始欄': '受傷程度', '2025原始欄': '受傷程度', '建議英文名': 'first_party_injury_severity_code', '中文定義': '單一當事者傷亡程度，不是事故傷亡總數，也無法直接辨識重傷', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉒；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；1 24小時內死亡、2 受傷、3 未受傷、4 不明、5 2–30日內死亡', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F34', '2024原始欄': '主要傷處', '2025原始欄': '主要傷處', '建議英文名': 'first_party_injury_part_code', '中文定義': '當事者主要受傷部位代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉓；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；09 多數傷、10 無、11 不明', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F35', '2024原始欄': '保護裝置', '2025原始欄': '保護裝置', '建議英文名': 'first_party_protection_code', '中文定義': '當事者保護裝備使用類別', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉔；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3 保護裝備；3 不明、4 無需使用等，版本映射仍需驗證', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F36', '2024原始欄': '行動電話', '2025原始欄': '行動電話', '建議英文名': 'first_party_phone_use_code', '中文定義': '行動電話、電腦或類似裝置使用情形', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉕；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；1 未使用、4 不明、5 非駕駛人', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F37', '2024原始欄': '車輛用途', '2025原始欄': '車輛用途', '建議英文名': 'first_party_vehicle_use_code', '中文定義': '車輛用途／非駕駛人類別', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉘；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；7 其他、8 非駕駛人及乘客', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F38', '2024原始欄': '當事者行動狀態', '2025原始欄': '當事者行動狀態', '建議英文名': 'first_party_action_code', '中文定義': '車或人的事故當時行動狀態', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉙；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；9 向前直行、16 步行、21 不明', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F39', '2024原始欄': '駕駛資格情形', '2025原始欄': '駕駛資格情形', '建議英文名': 'first_party_license_status_code', '中文定義': '駕照狀態代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉚；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；1 正常駕照、7 不明、8 無需駕照之人', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F40', '2024原始欄': '駕駛執照種類', '2025原始欄': '駕駛執照種類', '建議英文名': 'first_party_license_type_code', '中文定義': '駕照種類代碼', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉛；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；18 無駕照、19 不明、20 無需駕照之人', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F41', '2024原始欄': '飲酒情形', '2025原始欄': '飲酒情形', '建議英文名': 'first_party_alcohol_code', '中文定義': '酒精觀察／檢測分類，不是連續酒測值', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉜；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；1 觀察未飲酒、2 無反應、9 無法檢測、10 非駕駛人、11 不明', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F42', '2024原始欄': '車輛撞擊部位1', '2025原始欄': '車輛撞擊部位1', '建議英文名': 'first_party_initial_impact_code', '中文定義': '最初撞擊部位候選對應', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉝；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3 車輛撞擊部位 最初／其他；CSV 對應待發布規格確認', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F43', '2024原始欄': '車輛撞擊部位2', '2025原始欄': '不存在', '建議英文名': 'first_party_other_impact_code', '中文定義': '其他撞擊部位候選對應；僅 2024 欄', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉝；碼集見表8', '跨年分類': 'D', '可比較性': '不可直接建立兩年同口徑；2024主要肇因全缺失', '建議處理': '保留年度專屬欄；另一年為不存在，不補0、不混同普通缺失', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；與欄位 1 的發布規則待確認', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F44', '2024原始欄': '肇因碼-個別', '2025原始欄': '肇因碼-個別', '建議英文名': 'first_party_preliminary_cause_code', '中文定義': '初步分析研判之個別參照碼，不是法院責任或因果效應', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.4 初步分析研判索引；碼集見表8', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF pp.3–4 個別初步分析研判索引', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F45', '2024原始欄': '肇因碼-主要', '2025原始欄': '不存在', '建議英文名': 'main_preliminary_cause_code', '中文定義': '主要初步分析研判候選欄；僅 2024 且全缺失', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '事故主要研判欄（待認證）', '官方碼或單位': 'PDF p.4 初步分析研判索引；碼集見表8', '跨年分類': 'D', '可比較性': '不可直接建立兩年同口徑；2024主要肇因全缺失', '建議處理': '保留年度專屬欄；另一年為不存在，不補0、不混同普通缺失', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF pp.3–4；發布填值及跨年移除規則未知', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F46', '2024原始欄': '個人肇逃否', '2025原始欄': '個人肇逃否', '建議英文名': 'first_party_hit_run_code', '中文定義': '當事者肇事逃逸類別', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㉟；碼集見表8', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3；1 否、2 是', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F47', '2024原始欄': '職業', '2025原始欄': '職業', '建議英文名': 'first_party_occupation_code', '中文定義': '當事者職業代碼；本機碼與 PDF 不完整對應', '建議型態': 'Int64（nullable；保留原值）', '觀察層級': '第一當事人層級', '官方碼或單位': 'PDF p.3 ㊱；碼集見表8', '跨年分類': 'C', '可比較性': '條件式可比；定義或特殊值待確認', '建議處理': '保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併', '預計用途': 'E：不預計主分析', '定義依據及未知': 'PDF p.3 僅有部分項目；完整 CSV 職業碼表未知', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F48', '2024原始欄': '座標-X', '2025原始欄': '座標-X', '建議英文名': 'coordinate_x_source', '中文定義': '原始點位 X 數值；範圍近似經度但座標基準未確認', '建議型態': 'Float64（nullable；原始數值另保留）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '候選角度座標；CRS／基準未確認', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '身分／未來空間驗證輔助', '定義依據及未知': '官方頁僅稱 X、Y；不據範圍自行指定 EPSG', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}, {'field_id': 'F49', '2024原始欄': '座標-Y', '2025原始欄': '座標-Y', '建議英文名': 'coordinate_y_source', '中文定義': '原始點位 Y 數值；範圍近似緯度但座標基準未確認', '建議型態': 'Float64（nullable；原始數值另保留）', '觀察層級': '事故情境／總量層級（候選）', '官方碼或單位': '候選角度座標；CRS／基準未確認', '跨年分類': 'A', '可比較性': '原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致', '建議處理': '僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母', '預計用途': '身分／未來空間驗證輔助', '定義依據及未知': '官方頁僅稱 X、Y；不據範圍自行指定 EPSG', '是否預計主要分析': '否，僅補充或輔助；後續再審查'}]
schema_table=pd.DataFrame(SCHEMA)
assert set(schema_table['2024原始欄'])==set(raw[2024].columns)
assert set(schema_table.loc[schema_table['2025原始欄'].ne('不存在'),'2025原始欄'])==set(raw[2025].columns)
assert schema_table['2024原始欄'].is_unique
for label,columns in [
 ('欄位／型態／分類',['field_id','2024原始欄','2025原始欄','建議英文名','建議型態','跨年分類']),
 ('定義／觀察層級／來源',['field_id','2024原始欄','中文定義','觀察層級','官方碼或單位','定義依據及未知']),
 ('可比較性／用途／處理',['field_id','2024原始欄','可比較性','是否預計主要分析','預計用途','建議處理'])]:
    show('表 6：完整schema－'+label,schema_table[columns],block=15)
show('表 7：A–D可比較性分類與E研究關聯（非事故列數）',schema_table.groupby(['跨年分類','預計用途']).size().rename('欄位數').to_frame())
print({'mapped_2024_columns':len(raw[2024].columns),'mapped_2025_columns':len(raw[2025].columns),
       'mapping_rows':len(schema_table),'english_names_are_proposals':True})


**表 6：完整schema－欄位／型態／分類**

,field_id,2024原始欄,2025原始欄,建議英文名,建議型態,跨年分類
0,F01,發生年度,發生年度,accident_year,Int64（nullable；保留原值）,B
1,F02,發生月,發生月,accident_month,Int64（nullable；保留原值）,B
2,F03,發生日,發生日,accident_day,Int64（nullable；保留原值）,B
3,F04,發生時-Hours,發生時-Hours,accident_hour,Int64（nullable；保留原值）,B
4,F05,發生分,發生分,accident_minute,Int64（nullable；保留原值）,B
5,F06,區序,區序,district_label_source,string（保留原始碼／文字）,A
6,F07,肇事地點,肇事地點,accident_location_text,string（保留原始碼／文字）,A
7,F08,死亡人數,死亡人數,fatalities_24h_count_candidate,Int64（先查非負整數；原值另保留）,C
8,F09,2-30日死亡人數,2-30日死亡人數,fatalities_2_30d_count_candidate,Int64（先查非負整數；原值另保留）,C
9,F10,受傷人數,受傷人數,injured_count,Int64（先查非負整數；原值另保留）,A


,field_id,2024原始欄,2025原始欄,建議英文名,建議型態,跨年分類
15,F16,速限-速度限制,速限-速度限制,speed_limit_source,Float64（nullable；原始數值另保留）,C
16,F17,道路型態,道路型態,road_shape_code,Int64（nullable；保留原值）,A
17,F18,事故位置,事故位置,accident_position_code,Int64（nullable；保留原值）,A
18,F19,路面狀況1,路面狀況1,surface_material_code,Int64（nullable；保留原值）,A
19,F20,路面狀況2,路面狀況2,surface_condition_code,Int64（nullable；保留原值）,A
20,F21,路面狀況3,路面狀況3,surface_defect_code,Int64（nullable；保留原值）,A
21,F22,道路障礙1,道路障礙1,road_obstacle_code,Int64（nullable；保留原值）,A
22,F23,道路障礙2,道路障礙2,sight_condition_code,Int64（nullable；保留原值）,C
23,F24,號誌1,號誌1,signal_type_code,Int64（nullable；保留原值）,A
24,F25,號誌2,號誌2,signal_status_code,Int64（nullable；保留原值）,A


,field_id,2024原始欄,2025原始欄,建議英文名,建議型態,跨年分類
30,F31,性別,性別,first_party_sex_code,Int64（nullable；保留原值）,A
31,F32,年齡,年齡,first_party_age_source,Float64（nullable；原始數值另保留）,C
32,F33,受傷程度,受傷程度,first_party_injury_severity_code,Int64（nullable；保留原值）,A
33,F34,主要傷處,主要傷處,first_party_injury_part_code,Int64（nullable；保留原值）,A
34,F35,保護裝置,保護裝置,first_party_protection_code,Int64（nullable；保留原值）,A
35,F36,行動電話,行動電話,first_party_phone_use_code,Int64（nullable；保留原值）,A
36,F37,車輛用途,車輛用途,first_party_vehicle_use_code,Int64（nullable；保留原值）,A
37,F38,當事者行動狀態,當事者行動狀態,first_party_action_code,Int64（nullable；保留原值）,A
38,F39,駕駛資格情形,駕駛資格情形,first_party_license_status_code,Int64（nullable；保留原值）,A
39,F40,駕駛執照種類,駕駛執照種類,first_party_license_type_code,Int64（nullable；保留原值）,A


,field_id,2024原始欄,2025原始欄,建議英文名,建議型態,跨年分類
45,F46,個人肇逃否,個人肇逃否,first_party_hit_run_code,Int64（nullable；保留原值）,A
46,F47,職業,職業,first_party_occupation_code,Int64（nullable；保留原值）,C
47,F48,座標-X,座標-X,coordinate_x_source,Float64（nullable；原始數值另保留）,A
48,F49,座標-Y,座標-Y,coordinate_y_source,Float64（nullable；原始數值另保留）,A


**表 6：完整schema－定義／觀察層級／來源**

... (8 outputs left out)

,field_id,2024原始欄,可比較性,是否預計主要分析,預計用途,建議處理
45,F46,個人肇逃否,原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致,否，僅補充或輔助；後續再審查,E：不預計主分析,僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母
46,F47,職業,條件式可比；定義或特殊值待確認,否，僅補充或輔助；後續再審查,E：不預計主分析,保留原欄與版本；待來源確認語義，不以欄名或碼值相似直接合併
47,F48,座標-X,原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致,否，僅補充或輔助；後續再審查,身分／未來空間驗證輔助,僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母
48,F49,座標-Y,原始欄名相同、已核對觀測碼；未認證跨年發布產製完全一致,否，僅補充或輔助；後續再審查,身分／未來空間驗證輔助,僅建議統一欄名與nullable型態；不改原始值，缺失只影響所需變數分母


**表 7：A–D可比較性分類與E研究關聯（非事故列數）**

欄位數
跨年分類 預計用途              
A    A候選特徵輸入          2
     A候選特徵輸入；亦為C剖析    1
     E：不預計主分析        22
     補充變數             4
     身分／未來空間驗證輔助      5
B    A候選特徵輸入          5
C    C剖析候選            2
     E：不預計主分析         4
     補充變數             2
D    E：不預計主分析         2

{'mapped_2024_columns': 49, 'mapped_2025_columns': 47, 'mapping_rows': 49, 'english_names_are_proposals': True}


## 3. Official Codebook Validation
來源為[本機事故對照表](../raw/事故對照表.pdf) p.1、p.3、p.4（適用標示112.07.01後），
官方事故頁亦連到[同題名表單](https://www-ws.gov.taipei/001/Upload/467/relfile/32284/5040008/7ddb52e9-ebae-4834-b41a-5f41d5074279.pdf)。
本次已讀取官方表單頁面並目視本機 p.1、p.3；未替換原始 PDF，未宣稱兩者位元組相同。
表單同時印列部分新舊碼；「印列碼集」不能自動認證兩年 CSV 的發布映射。

主特徵所需型態1–9／13／16、位置1–4，以及天候、道路型態、車種、性別均列逐碼定義。
其他未預計主分析的欄位列可核對碼集及頁／欄索引；未另補造 PDF 沒有的職業等碼義。
「特殊／不明／不適用」是可合法記錄的狀態，不等同原始空白或錯誤。
碼表完整性未確認者，其碼外筆數列為「未證實」，確認無效數留 NA，不能當作 0 或直接刪除。


In [5]:
BOOKS = {'accident_type': {'columns': {2024: '事故類型及型態', 2025: '事故類型及型態'}, 'labels': {1: '對向通行中', 2: '同向通行中', 3: '穿越道路中', 4: '在路上嬉戲', 5: '在路上作業中', 6: '衝進路中', 7: '從停車後或中穿出', 8: '佇立路邊外', 9: '人與車其他', 10: '對撞', 11: '對向擦撞', 12: '同向擦撞', 13: '追撞', 14: '倒車撞', 15: '路口交岔撞', 16: '側撞', 17: '車與車其他', 18: '路上翻車摔倒', 19: '衝出路外', 20: '撞護欄樁', 21: '撞號誌標誌桿', 22: '撞收費亭', 23: '撞交通島', 24: '撞非固定設施', 25: '撞橋梁建築物', 26: '撞路樹電桿', 27: '撞動物', 28: '撞工程施工', 29: '車輛本身其他', 30: '衝過或撞壞遮斷器', 31: '正越過平交道中', 32: '暫停位置不當', 33: '平交道內無法行動', 34: '平交道其他', 35: '撞橋樑橋墩', 36: '撞建築物', 37: '撞路樹', 38: '撞電桿'}, 'page': 1, 'section': '⑮；車與車定義另見p.2', 'special': {9: '人車其他，仍屬人與車', 17: '車車其他，仍為有效類別', 29: '車輛本身其他', 34: '平交道其他'}, 'complete': True, 'caveat': '表單同列部分新舊碼；未觀測舊碼不推定已完成版本認證'}, 'position': {'columns': {2024: '事故位置', 2025: '事故位置'}, 'labels': {1: '交岔路口內', 2: '交岔口附近', 3: '機慢車待轉區', 4: '機慢車停等區', 5: '交通島含槽化線', 6: '迴轉道', 7: '快車道', 8: '慢車道', 9: '一般車道', 10: '公車專用道', 11: '機車專用道', 12: '機車優先道', 13: '路肩路緣', 14: '加速車道', 15: '減速車道', 16: '直線匝道', 17: '環道匝道', 18: '行人穿越道', 19: '行人穿越道附近', 20: '人行道', 22: '其他', 23: '自行車專用道', 24: '人行道標線', 25: '騎樓'}, 'page': 1, 'section': '⑨', 'special': {}, 'complete': True, 'caveat': ''}, 'road_shape': {'columns': {2024: '道路型態', 2025: '道路型態'}, 'labels': {1: '平交道有遮斷器', 2: '平交道無遮斷器', 3: '三岔路', 4: '四岔路', 5: '多岔路', 6: '隧道', 7: '地下道', 8: '橋樑', 9: '涵洞', 10: '高架道路', 11: '彎曲路及附近', 12: '坡路', 13: '巷弄', 14: '直路', 18: '圓環', 19: '廣場', 20: '休息站或服務區', 21: '輕軌共用道路', 22: '其他'}, 'page': 1, 'section': '⑧', 'special': {}, 'complete': True, 'caveat': ''}, 'weather': {'columns': {2024: '天候', 2025: '天候'}, 'labels': {1: '暴雨', 2: '強風', 3: '風沙', 4: '霧或煙', 5: '雪', 6: '雨', 7: '陰', 8: '晴', 9: '風'}, 'page': 1, 'section': '④', 'special': {}, 'complete': True, 'caveat': ''}, 'lighting': {'columns': {2024: '道路照明設備', 2025: '光線'}, 'labels': {1: '日間自然光線', 2: '晨或暮光', 3: '夜間隧道等有照明', 4: '夜間隧道等無照明', 5: '有照明且開啟', 6: '有照明未開啟或故障', 7: '無照明'}, 'page': 1, 'section': '⑤', 'special': {}, 'complete': False, 'caveat': '表單同列1–4與5–7；兩年欄位含義及代碼版本等價尚未證實，僅核對印列值'}, 'vehicle': {'columns': {2024: '車種', 2025: '車種'}, 'labels': {'A01': '公營公車', 'A02': '民營公車', 'A03': '公營客運', 'A04': '民營客運', 'A05': '遊覽車', 'A06': '自用大客車', 'A11': '營業大貨車', 'A12': '自用大貨車', 'A21': '營業全聯結車', 'A22': '自用全聯結車', 'A31': '營業半聯結車', 'A32': '自用半聯結車', 'A41': '營業曳引車', 'A42': '自用曳引車', 'B01': '計程車', 'B02': '租賃小客車', 'B03': '自用小客車', 'B11': '營業小貨車', 'B12': '自用小貨車', 'B13': '租賃小貨車', 'C01': '大型重型機車550cc以上', 'C02': '大型重型機車250至550cc', 'C03': '普通重型機車', 'C04': '普通輕型機車', 'C05': '小型輕型機車', 'D01': '軍用大客車', 'D02': '軍用載重車', 'D03': '軍用小型車', 'E01': '救護車', 'E02': '消防車', 'E03': '警備車', 'E04': '工程車', 'E05': '其他特種車', 'F01': '腳踏自行車', 'F02': '電動輔助自行車', 'F03': '微型電動二輪車', 'F04': '人力車', 'F05': '獸力車', 'F06': '其他慢車', 'F07': '個人行動運具', 'G01': '拼裝車', 'G02': '農耕用車或機械', 'G03': '動力機械', 'G04': '拖車架', 'G05': '火車', 'G06': '其他車', 'G07': '大眾捷運系統車輛', 'H01': '行人', 'H02': '乘客', 'H03': '其他人', 'H04': '輔助代步器材'}, 'page': 3, 'section': '㉖', 'special': {'H01': '行人，不是車輛', 'H02': '乘客', 'H03': '其他人', 'H04': '輔助代步器材'}, 'complete': True, 'caveat': ''}, 'sex': {'columns': {2024: '性別', 2025: '性別'}, 'labels': {1: '男', 2: '女', 3: '無或物／動物／堆置物', 4: '肇逃尚未查獲', 5: '無人駕駛自動車'}, 'page': 3, 'section': '⑰', 'special': {3: '非男女自然人', 4: '未查獲', 5: '無人駕駛'}, 'complete': True, 'caveat': ''}, 'party_injury': {'columns': {2024: '受傷程度', 2025: '受傷程度'}, 'labels': {1: '24小時內死亡', 2: '受傷', 3: '未受傷', 4: '不明', 5: '2至30日內死亡'}, 'page': 3, 'section': '㉒', 'special': {4: '不明'}, 'complete': True, 'caveat': ''}, 'road_class': {'columns': {2024: '道路類別', 2025: '道路類別'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑥', 'special': {}, 'complete': True, 'caveat': ''}, 'surface_material': {'columns': {2024: '路面狀況1', 2025: '路面狀況1'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑩(1)', 'special': {}, 'complete': True, 'caveat': ''}, 'surface_condition': {'columns': {2024: '路面狀況2', 2025: '路面狀況2'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑩(2)', 'special': {}, 'complete': True, 'caveat': ''}, 'surface_defect': {'columns': {2024: '路面狀況3', 2025: '路面狀況3'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑩(3)', 'special': {}, 'complete': True, 'caveat': ''}, 'obstacle': {'columns': {2024: '道路障礙1', 2025: '道路障礙1'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑪(1)', 'special': {}, 'complete': True, 'caveat': ''}, 'signal_type': {'columns': {2024: '號誌1', 2025: '號誌1'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑫(1)', 'special': {4: '無號誌，非缺失'}, 'complete': True, 'caveat': ''}, 'signal_status': {'columns': {2024: '號誌2', 2025: '號誌2'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑫(2)', 'special': {4: '無號誌，非缺失'}, 'complete': True, 'caveat': ''}, 'direction_divider': {'columns': {2024: '車道劃分-分向', 2025: '車道劃分-分向'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑬', 'special': {}, 'complete': True, 'caveat': ''}, 'lane_general': {'columns': {2024: '車道劃分-分道1', 2025: '車道劃分-分道1'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑭(1)', 'special': {}, 'complete': True, 'caveat': ''}, 'lane_fast_slow': {'columns': {2024: '車道劃分-分道2', 2025: '車道劃分-分道2'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑭(2)', 'special': {}, 'complete': True, 'caveat': ''}, 'road_edge': {'columns': {2024: '車道劃分-分道3', 2025: '車道劃分-分道3'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄'}, 'page': 1, 'section': '⑭(3)', 'special': {}, 'complete': True, 'caveat': ''}, 'injury_part': {'columns': {2024: '主要傷處', 2025: '主要傷處'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄', 11: '表單碼11；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉓', 'special': {10: '無', 11: '不明'}, 'complete': True, 'caveat': ''}, 'protection': {'columns': {2024: '保護裝置', 2025: '保護裝置'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉔', 'special': {3: '不明', 4: '其他／無需使用'}, 'complete': True, 'caveat': ''}, 'phone': {'columns': {2024: '行動電話', 2025: '行動電話'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉕', 'special': {4: '不明', 5: '非駕駛人'}, 'complete': True, 'caveat': ''}, 'vehicle_use': {'columns': {2024: '車輛用途', 2025: '車輛用途'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉘', 'special': {8: '非駕駛人及乘客'}, 'complete': True, 'caveat': ''}, 'action': {'columns': {2024: '當事者行動狀態', 2025: '當事者行動狀態'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄', 11: '表單碼11；語意明細見指定頁／欄', 12: '表單碼12；語意明細見指定頁／欄', 13: '表單碼13；語意明細見指定頁／欄', 14: '表單碼14；語意明細見指定頁／欄', 15: '表單碼15；語意明細見指定頁／欄', 16: '表單碼16；語意明細見指定頁／欄', 17: '表單碼17；語意明細見指定頁／欄', 18: '表單碼18；語意明細見指定頁／欄', 19: '表單碼19；語意明細見指定頁／欄', 20: '表單碼20；語意明細見指定頁／欄', 21: '表單碼21；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉙', 'special': {21: '不明'}, 'complete': True, 'caveat': ''}, 'license_status': {'columns': {2024: '駕駛資格情形', 2025: '駕駛資格情形'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉚', 'special': {7: '不明', 8: '無需駕照'}, 'complete': True, 'caveat': ''}, 'license_type': {'columns': {2024: '駕駛執照種類', 2025: '駕駛執照種類'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄', 11: '表單碼11；語意明細見指定頁／欄', 12: '表單碼12；語意明細見指定頁／欄', 13: '表單碼13；語意明細見指定頁／欄', 14: '表單碼14；語意明細見指定頁／欄', 15: '表單碼15；語意明細見指定頁／欄', 16: '表單碼16；語意明細見指定頁／欄', 17: '表單碼17；語意明細見指定頁／欄', 18: '表單碼18；語意明細見指定頁／欄', 19: '表單碼19；語意明細見指定頁／欄', 20: '表單碼20；語意明細見指定頁／欄', 21: '表單碼21；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉛', 'special': {18: '無駕照', 19: '不明', 20: '無需駕照'}, 'complete': True, 'caveat': ''}, 'alcohol': {'columns': {2024: '飲酒情形', 2025: '飲酒情形'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄', 11: '表單碼11；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉜', 'special': {9: '無法檢測', 10: '非駕駛人未檢測', 11: '不明'}, 'complete': True, 'caveat': ''}, 'hit_first': {'columns': {2024: '車輛撞擊部位1', 2025: '車輛撞擊部位1'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄', 11: '表單碼11；語意明細見指定頁／欄', 12: '表單碼12；語意明細見指定頁／欄', 13: '表單碼13；語意明細見指定頁／欄', 14: '表單碼14；語意明細見指定頁／欄', 15: '表單碼15；語意明細見指定頁／欄', 16: '表單碼16；語意明細見指定頁／欄', 17: '表單碼17；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉝', 'special': {15: '不明', 16: '非汽機自行車', 17: '無撞擊'}, 'complete': True, 'caveat': ''}, 'hit_other': {'columns': {2024: '車輛撞擊部位2', 2025: None}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄', 3: '表單碼3；語意明細見指定頁／欄', 4: '表單碼4；語意明細見指定頁／欄', 5: '表單碼5；語意明細見指定頁／欄', 6: '表單碼6；語意明細見指定頁／欄', 7: '表單碼7；語意明細見指定頁／欄', 8: '表單碼8；語意明細見指定頁／欄', 9: '表單碼9；語意明細見指定頁／欄', 10: '表單碼10；語意明細見指定頁／欄', 11: '表單碼11；語意明細見指定頁／欄', 12: '表單碼12；語意明細見指定頁／欄', 13: '表單碼13；語意明細見指定頁／欄', 14: '表單碼14；語意明細見指定頁／欄', 15: '表單碼15；語意明細見指定頁／欄', 16: '表單碼16；語意明細見指定頁／欄', 17: '表單碼17；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉝', 'special': {15: '不明', 16: '非汽機自行車', 17: '無撞擊'}, 'complete': True, 'caveat': ''}, 'hit_run': {'columns': {2024: '個人肇逃否', 2025: '個人肇逃否'}, 'labels': {1: '表單碼1；語意明細見指定頁／欄', 2: '表單碼2；語意明細見指定頁／欄'}, 'page': 3, 'section': '㉟', 'special': {}, 'complete': True, 'caveat': ''}, 'sight': {'columns': {2024: '道路障礙2', 2025: '道路障礙2'}, 'labels': {1: '視距印列碼1；見PDF⑪(2)', 2: '視距印列碼2；見PDF⑪(2)', 3: '視距印列碼3；見PDF⑪(2)', 4: '視距印列碼4；見PDF⑪(2)', 5: '視距印列碼5；見PDF⑪(2)', 6: '視距印列碼6；見PDF⑪(2)', 7: '視距印列碼7；見PDF⑪(2)', 8: '視距印列碼8；見PDF⑪(2)', 9: '視距印列碼9；見PDF⑪(2)', 10: '視距印列碼10；見PDF⑪(2)', 11: '視距印列碼11；見PDF⑪(2)', 12: '視距印列碼12；見PDF⑪(2)', 13: '視距印列碼13；見PDF⑪(2)'}, 'page': 1, 'section': '⑪(2)', 'special': {}, 'complete': False, 'caveat': '同列1–7與8–13，不能逕認所有碼同一版本'}, 'cause': {'columns': {2024: '肇因碼-個別', 2025: '肇因碼-個別'}, 'labels': {1: '初步研判索引1；逐碼語意見PDF p.4', 2: '初步研判索引2；逐碼語意見PDF p.4', 4: '初步研判索引4；逐碼語意見PDF p.4', 5: '初步研判索引5；逐碼語意見PDF p.4', 8: '初步研判索引8；逐碼語意見PDF p.4', 9: '初步研判索引9；逐碼語意見PDF p.4', 10: '初步研判索引10；逐碼語意見PDF p.4', 11: '初步研判索引11；逐碼語意見PDF p.4', 12: '初步研判索引12；逐碼語意見PDF p.4', 13: '初步研判索引13；逐碼語意見PDF p.4', 14: '初步研判索引14；逐碼語意見PDF p.4', 16: '初步研判索引16；逐碼語意見PDF p.4', 17: '初步研判索引17；逐碼語意見PDF p.4', 18: '初步研判索引18；逐碼語意見PDF p.4', 19: '初步研判索引19；逐碼語意見PDF p.4', 20: '初步研判索引20；逐碼語意見PDF p.4', 21: '初步研判索引21；逐碼語意見PDF p.4', 22: '初步研判索引22；逐碼語意見PDF p.4', 24: '初步研判索引24；逐碼語意見PDF p.4', 27: '初步研判索引27；逐碼語意見PDF p.4', 28: '初步研判索引28；逐碼語意見PDF p.4', 29: '初步研判索引29；逐碼語意見PDF p.4', 30: '初步研判索引30；逐碼語意見PDF p.4', 31: '初步研判索引31；逐碼語意見PDF p.4', 32: '初步研判索引32；逐碼語意見PDF p.4', 33: '初步研判索引33；逐碼語意見PDF p.4', 34: '初步研判索引34；逐碼語意見PDF p.4', 35: '初步研判索引35；逐碼語意見PDF p.4', 36: '初步研判索引36；逐碼語意見PDF p.4', 37: '初步研判索引37；逐碼語意見PDF p.4', 38: '初步研判索引38；逐碼語意見PDF p.4', 39: '初步研判索引39；逐碼語意見PDF p.4', 40: '初步研判索引40；逐碼語意見PDF p.4', 42: '初步研判索引42；逐碼語意見PDF p.4', 44: '初步研判索引44；逐碼語意見PDF p.4', 45: '初步研判索引45；逐碼語意見PDF p.4', 46: '初步研判索引46；逐碼語意見PDF p.4', 47: '初步研判索引47；逐碼語意見PDF p.4', 48: '初步研判索引48；逐碼語意見PDF p.4', 49: '初步研判索引49；逐碼語意見PDF p.4', 50: '初步研判索引50；逐碼語意見PDF p.4', 53: '初步研判索引53；逐碼語意見PDF p.4', 54: '初步研判索引54；逐碼語意見PDF p.4', 55: '初步研判索引55；逐碼語意見PDF p.4', 56: '初步研判索引56；逐碼語意見PDF p.4', 57: '初步研判索引57；逐碼語意見PDF p.4', 58: '初步研判索引58；逐碼語意見PDF p.4', 59: '初步研判索引59；逐碼語意見PDF p.4', 60: '初步研判索引60；逐碼語意見PDF p.4', 61: '初步研判索引61；逐碼語意見PDF p.4', 62: '初步研判索引62；逐碼語意見PDF p.4', 64: '初步研判索引64；逐碼語意見PDF p.4', 65: '初步研判索引65；逐碼語意見PDF p.4', 66: '初步研判索引66；逐碼語意見PDF p.4', 67: '初步研判索引67；逐碼語意見PDF p.4', 68: '初步研判索引68；逐碼語意見PDF p.4', 69: '初步研判索引69；逐碼語意見PDF p.4', 70: '初步研判索引70；逐碼語意見PDF p.4', 71: '初步研判索引71；逐碼語意見PDF p.4', 72: '初步研判索引72；逐碼語意見PDF p.4', 73: '初步研判索引73；逐碼語意見PDF p.4', 74: '初步研判索引74；逐碼語意見PDF p.4', 75: '初步研判索引75；逐碼語意見PDF p.4', 76: '初步研判索引76；逐碼語意見PDF p.4', 77: '初步研判索引77；逐碼語意見PDF p.4', 78: '初步研判索引78；逐碼語意見PDF p.4', 79: '初步研判索引79；逐碼語意見PDF p.4', 80: '初步研判索引80；逐碼語意見PDF p.4', 81: '初步研判索引81；逐碼語意見PDF p.4', 82: '初步研判索引82；逐碼語意見PDF p.4', 83: '初步研判索引83；逐碼語意見PDF p.4', 84: '初步研判索引84；逐碼語意見PDF p.4', 85: '初步研判索引85；逐碼語意見PDF p.4', 86: '初步研判索引86；逐碼語意見PDF p.4', 87: '初步研判索引87；逐碼語意見PDF p.4', 88: '初步研判索引88；逐碼語意見PDF p.4', 89: '初步研判索引89；逐碼語意見PDF p.4', 90: '初步研判索引90；逐碼語意見PDF p.4', 91: '初步研判索引91；逐碼語意見PDF p.4', 92: '初步研判索引92；逐碼語意見PDF p.4', 93: '初步研判索引93；逐碼語意見PDF p.4', 94: '初步研判索引94；逐碼語意見PDF p.4', 95: '初步研判索引95；逐碼語意見PDF p.4', 96: '初步研判索引96；逐碼語意見PDF p.4', 97: '初步研判索引97；逐碼語意見PDF p.4', 98: '初步研判索引98；逐碼語意見PDF p.4', 99: '初步研判索引99；逐碼語意見PDF p.4', 100: '初步研判索引100；逐碼語意見PDF p.4', 101: '初步研判索引101；逐碼語意見PDF p.4', 102: '初步研判索引102；逐碼語意見PDF p.4', 103: '初步研判索引103；逐碼語意見PDF p.4', 104: '初步研判索引104；逐碼語意見PDF p.4', 105: '初步研判索引105；逐碼語意見PDF p.4', 106: '初步研判索引106；逐碼語意見PDF p.4', 107: '初步研判索引107；逐碼語意見PDF p.4', 108: '初步研判索引108；逐碼語意見PDF p.4', 109: '初步研判索引109；逐碼語意見PDF p.4', 110: '初步研判索引110；逐碼語意見PDF p.4', 111: '初步研判索引111；逐碼語意見PDF p.4', 112: '初步研判索引112；逐碼語意見PDF p.4', 113: '初步研判索引113；逐碼語意見PDF p.4', 114: '初步研判索引114；逐碼語意見PDF p.4', 115: '初步研判索引115；逐碼語意見PDF p.4', 116: '初步研判索引116；逐碼語意見PDF p.4', 117: '初步研判索引117；逐碼語意見PDF p.4', 118: '初步研判索引118；逐碼語意見PDF p.4', 119: '初步研判索引119；逐碼語意見PDF p.4', 120: '初步研判索引120；逐碼語意見PDF p.4', 121: '初步研判索引121；逐碼語意見PDF p.4', 122: '初步研判索引122；逐碼語意見PDF p.4', 123: '初步研判索引123；逐碼語意見PDF p.4', 124: '初步研判索引124；逐碼語意見PDF p.4', 125: '初步研判索引125；逐碼語意見PDF p.4', 126: '初步研判索引126；逐碼語意見PDF p.4', 127: '初步研判索引127；逐碼語意見PDF p.4', 128: '初步研判索引128；逐碼語意見PDF p.4', 129: '初步研判索引129；逐碼語意見PDF p.4', 130: '初步研判索引130；逐碼語意見PDF p.4', 131: '初步研判索引131；逐碼語意見PDF p.4'}, 'page': 4, 'section': '初步分析研判索引', 'special': {44: '尚未發現肇事因素', 67: '尚未發現肇事因素', 118: '跡證不足無法釐清', 119: '肇逃未查獲無法查明'}, 'complete': False, 'caveat': '可辨識印列碼集；版本／發布對應未認證，碼外先列未證實，不當官方無效'}, 'cause_main': {'columns': {2024: '肇因碼-主要', 2025: None}, 'labels': {1: '初步研判索引1；見PDF p.4', 2: '初步研判索引2；見PDF p.4', 4: '初步研判索引4；見PDF p.4', 5: '初步研判索引5；見PDF p.4', 8: '初步研判索引8；見PDF p.4', 9: '初步研判索引9；見PDF p.4', 10: '初步研判索引10；見PDF p.4', 11: '初步研判索引11；見PDF p.4', 12: '初步研判索引12；見PDF p.4', 13: '初步研判索引13；見PDF p.4', 14: '初步研判索引14；見PDF p.4', 16: '初步研判索引16；見PDF p.4', 17: '初步研判索引17；見PDF p.4', 18: '初步研判索引18；見PDF p.4', 19: '初步研判索引19；見PDF p.4', 20: '初步研判索引20；見PDF p.4', 21: '初步研判索引21；見PDF p.4', 22: '初步研判索引22；見PDF p.4', 24: '初步研判索引24；見PDF p.4', 27: '初步研判索引27；見PDF p.4', 28: '初步研判索引28；見PDF p.4', 29: '初步研判索引29；見PDF p.4', 30: '初步研判索引30；見PDF p.4', 31: '初步研判索引31；見PDF p.4', 32: '初步研判索引32；見PDF p.4', 33: '初步研判索引33；見PDF p.4', 34: '初步研判索引34；見PDF p.4', 35: '初步研判索引35；見PDF p.4', 36: '初步研判索引36；見PDF p.4', 37: '初步研判索引37；見PDF p.4', 38: '初步研判索引38；見PDF p.4', 39: '初步研判索引39；見PDF p.4', 40: '初步研判索引40；見PDF p.4', 42: '初步研判索引42；見PDF p.4', 44: '初步研判索引44；見PDF p.4', 45: '初步研判索引45；見PDF p.4', 46: '初步研判索引46；見PDF p.4', 47: '初步研判索引47；見PDF p.4', 48: '初步研判索引48；見PDF p.4', 49: '初步研判索引49；見PDF p.4', 50: '初步研判索引50；見PDF p.4', 53: '初步研判索引53；見PDF p.4', 54: '初步研判索引54；見PDF p.4', 55: '初步研判索引55；見PDF p.4', 56: '初步研判索引56；見PDF p.4', 57: '初步研判索引57；見PDF p.4', 58: '初步研判索引58；見PDF p.4', 59: '初步研判索引59；見PDF p.4', 60: '初步研判索引60；見PDF p.4', 61: '初步研判索引61；見PDF p.4', 62: '初步研判索引62；見PDF p.4', 64: '初步研判索引64；見PDF p.4', 65: '初步研判索引65；見PDF p.4', 66: '初步研判索引66；見PDF p.4', 67: '初步研判索引67；見PDF p.4', 68: '初步研判索引68；見PDF p.4', 69: '初步研判索引69；見PDF p.4', 70: '初步研判索引70；見PDF p.4', 71: '初步研判索引71；見PDF p.4', 72: '初步研判索引72；見PDF p.4', 73: '初步研判索引73；見PDF p.4', 74: '初步研判索引74；見PDF p.4', 75: '初步研判索引75；見PDF p.4', 76: '初步研判索引76；見PDF p.4', 77: '初步研判索引77；見PDF p.4', 78: '初步研判索引78；見PDF p.4', 79: '初步研判索引79；見PDF p.4', 80: '初步研判索引80；見PDF p.4', 81: '初步研判索引81；見PDF p.4', 82: '初步研判索引82；見PDF p.4', 83: '初步研判索引83；見PDF p.4', 84: '初步研判索引84；見PDF p.4', 85: '初步研判索引85；見PDF p.4', 86: '初步研判索引86；見PDF p.4', 87: '初步研判索引87；見PDF p.4', 88: '初步研判索引88；見PDF p.4', 89: '初步研判索引89；見PDF p.4', 90: '初步研判索引90；見PDF p.4', 91: '初步研判索引91；見PDF p.4', 92: '初步研判索引92；見PDF p.4', 93: '初步研判索引93；見PDF p.4', 94: '初步研判索引94；見PDF p.4', 95: '初步研判索引95；見PDF p.4', 96: '初步研判索引96；見PDF p.4', 97: '初步研判索引97；見PDF p.4', 98: '初步研判索引98；見PDF p.4', 99: '初步研判索引99；見PDF p.4', 100: '初步研判索引100；見PDF p.4', 101: '初步研判索引101；見PDF p.4', 102: '初步研判索引102；見PDF p.4', 103: '初步研判索引103；見PDF p.4', 104: '初步研判索引104；見PDF p.4', 105: '初步研判索引105；見PDF p.4', 106: '初步研判索引106；見PDF p.4', 107: '初步研判索引107；見PDF p.4', 108: '初步研判索引108；見PDF p.4', 109: '初步研判索引109；見PDF p.4', 110: '初步研判索引110；見PDF p.4', 111: '初步研判索引111；見PDF p.4', 112: '初步研判索引112；見PDF p.4', 113: '初步研判索引113；見PDF p.4', 114: '初步研判索引114；見PDF p.4', 115: '初步研判索引115；見PDF p.4', 116: '初步研判索引116；見PDF p.4', 117: '初步研判索引117；見PDF p.4', 118: '初步研判索引118；見PDF p.4', 119: '初步研判索引119；見PDF p.4', 120: '初步研判索引120；見PDF p.4', 121: '初步研判索引121；見PDF p.4', 122: '初步研判索引122；見PDF p.4', 123: '初步研判索引123；見PDF p.4', 124: '初步研判索引124；見PDF p.4', 125: '初步研判索引125；見PDF p.4', 126: '初步研判索引126；見PDF p.4', 127: '初步研判索引127；見PDF p.4', 128: '初步研判索引128；見PDF p.4', 129: '初步研判索引129；見PDF p.4', 130: '初步研判索引130；見PDF p.4', 131: '初步研判索引131；見PDF p.4'}, 'page': 4, 'section': '初步分析研判索引', 'special': {}, 'complete': False, 'caveat': '2024全缺失、2025無此欄；不可補0'}, 'occupation': {'columns': {2024: '職業', 2025: '職業'}, 'labels': {14: '小學生', 15: '國中生', 16: '高中職生', 24: '6歲以下嬰幼童', 25: '大學生', 26: '勞工人員'}, 'page': 3, 'section': '㊱', 'special': {}, 'complete': False, 'caveat': 'PDF僅部分職業碼；21／22等大量未印列值不能逕判錯誤'}}
book_summary=[];definitions=[];code_audit=[];code_distributions={}
for key,book in BOOKS.items():
    allowed=set(book['labels'])
    book_summary.append({'變數':key,'2024欄':book['columns'][2024],'2025欄':book['columns'][2025] or '不存在',
       'PDF頁':book['page'],'表單欄':book['section'],'可核對印列碼':', '.join(map(str,sorted(allowed))),
       '足以檢查碼外':book['complete'],'特殊碼':str(book['special']),'限制':book['caveat']})
    if key in ['accident_type','position','road_shape','weather','lighting','vehicle','sex','party_injury']:
        for value,label in book['labels'].items():
            definitions.append({'變數':key,'碼':str(value),'表單定義':label,'PDF頁':book['page'],'表單欄':book['section'],
                                '備註':book['special'].get(value,book['caveat'])})
    rows=[]
    for year,frame in candidates.items():
        column=book['columns'][year]
        if column is None:continue
        s=frame[column]
        outside=s.notna()&~s.isin(allowed)
        code_audit.append({'year':year,'變數':key,'原欄':column,'候選N':len(frame),'缺失':int(s.isna().sum()),
            '非缺失碼外':int(outside.sum()),'碼外判定':'未在本次印列集／未證實' if not book['complete'] else '不符本次可核對表單碼集',
            '可核對碼外數':int(outside.sum()) if book['complete'] else pd.NA,
            '特殊碼列數':int(s.isin(book['special']).sum()),'PDF頁':book['page']})
        for value,count in s.value_counts(dropna=False).sort_index().items():
            missing=pd.isna(value)
            rows.append({'year':year,'碼':'NA（原始缺失）' if missing else str(value),
              '原始碼定義':'原始缺失' if missing else book['labels'].get(value,'本次碼表未列／定義待補'),
              '候選筆數':int(count),'占全部候選%':100*count/len(frame),
              '占本欄非缺失%':pd.NA if missing else 100*count/s.notna().sum(),
              '狀態':'缺失' if missing else ('印列特殊碼' if value in book['special'] else ('印列碼' if value in allowed else '碼外／未證實'))})
    code_distributions[key]=pd.DataFrame(rows)
show('表 8：官方印列碼集、來源頁碼與完整性限制',pd.DataFrame(book_summary),block=10)
show('表 9：主要與補充變數逐碼定義',pd.DataFrame(definitions),block=25)
code_audit_table=pd.DataFrame(code_audit)
show('表 10：逐年碼表核對、缺失與特殊碼摘要',code_audit_table,block=20)
for key,table in code_distributions.items():
    show('表 11：'+key+' 兩年實際碼分布（逐年分母，不檢定差異）',table,block=30)


**表 8：官方印列碼集、來源頁碼與完整性限制**

,變數,2024欄,2025欄,PDF頁,表單欄,可核對印列碼,足以檢查碼外,特殊碼,限制
0,accident_type,事故類型及型態,事故類型及型態,1,⑮；車與車定義另見p.2,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29,...",True,"{9: '人車其他，仍屬人與車', 17: '車車其他，仍為有效類別', 29: '車輛本身其他', 34: '平交道其他'}",表單同列部分新舊碼；未觀測舊碼不推定已完成版本認證
1,position,事故位置,事故位置,1,⑨,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25",True,{},
2,road_shape,道路型態,道路型態,1,⑧,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 18, 19, 20, 21, 22",True,{},
3,weather,天候,天候,1,④,"1, 2, 3, 4, 5, 6, 7, 8, 9",True,{},
4,lighting,道路照明設備,光線,1,⑤,"1, 2, 3, 4, 5, 6, 7",False,{},表單同列1–4與5–7；兩年欄位含義及代碼版本等價尚未證實，僅核對印列值
5,vehicle,車種,車種,3,㉖,"A01, A02, A03, A04, A05, A06, A11, A12, A21, A22, A31, A32, A41, A42, B01, B02, B03, B11, B12, B13, C01, C...",True,"{'H01': '行人，不是車輛', 'H02': '乘客', 'H03': '其他人', 'H04': '輔助代步器材'}",
6,sex,性別,性別,3,⑰,"1, 2, 3, 4, 5",True,"{3: '非男女自然人', 4: '未查獲', 5: '無人駕駛'}",
7,party_injury,受傷程度,受傷程度,3,㉒,"1, 2, 3, 4, 5",True,{4: '不明'},
8,road_class,道路類別,道路類別,1,⑥,"1, 2, 3, 4, 5, 6, 7, 8, 9",True,{},
9,surface_material,路面狀況1,路面狀況1,1,⑩(1),"1, 2, 3, 4, 5",True,{},


,變數,2024欄,2025欄,PDF頁,表單欄,可核對印列碼,足以檢查碼外,特殊碼,限制
10,surface_condition,路面狀況2,路面狀況2,1,⑩(2),"1, 2, 3, 4, 5",True,{},
11,surface_defect,路面狀況3,路面狀況3,1,⑩(3),"1, 3, 4, 5",True,{},
12,obstacle,道路障礙1,道路障礙1,1,⑪(1),"1, 2, 3, 4, 5, 6",True,{},
13,signal_type,號誌1,號誌1,1,⑫(1),"1, 2, 3, 4",True,{4: '無號誌，非缺失'},
14,signal_status,號誌2,號誌2,1,⑫(2),"1, 2, 3, 4",True,{4: '無號誌，非缺失'},
15,direction_divider,車道劃分-分向,車道劃分-分向,1,⑬,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10",True,{},
16,lane_general,車道劃分-分道1,車道劃分-分道1,1,⑭(1),"1, 2, 3, 4, 5",True,{},
17,lane_fast_slow,車道劃分-分道2,車道劃分-分道2,1,⑭(2),"1, 2, 3, 4, 5",True,{},
18,road_edge,車道劃分-分道3,車道劃分-分道3,1,⑭(3),"1, 2",True,{},
19,injury_part,主要傷處,主要傷處,3,㉓,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11",True,"{10: '無', 11: '不明'}",


,變數,2024欄,2025欄,PDF頁,表單欄,可核對印列碼,足以檢查碼外,特殊碼,限制
20,protection,保護裝置,保護裝置,3,㉔,"1, 2, 3, 4, 5, 6, 7",True,"{3: '不明', 4: '其他／無需使用'}",
21,phone,行動電話,行動電話,3,㉕,"1, 2, 3, 4, 5",True,"{4: '不明', 5: '非駕駛人'}",
22,vehicle_use,車輛用途,車輛用途,3,㉘,"1, 2, 3, 4, 5, 6, 7, 8",True,{8: '非駕駛人及乘客'},
23,action,當事者行動狀態,當事者行動狀態,3,㉙,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21",True,{21: '不明'},
24,license_status,駕駛資格情形,駕駛資格情形,3,㉚,"1, 2, 3, 4, 5, 6, 7, 8",True,"{7: '不明', 8: '無需駕照'}",
25,license_type,駕駛執照種類,駕駛執照種類,3,㉛,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21",True,"{18: '無駕照', 19: '不明', 20: '無需駕照'}",
26,alcohol,飲酒情形,飲酒情形,3,㉜,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11",True,"{9: '無法檢測', 10: '非駕駛人未檢測', 11: '不明'}",
27,hit_first,車輛撞擊部位1,車輛撞擊部位1,3,㉝,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17",True,"{15: '不明', 16: '非汽機自行車', 17: '無撞擊'}",
28,hit_other,車輛撞擊部位2,不存在,3,㉝,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17",True,"{15: '不明', 16: '非汽機自行車', 17: '無撞擊'}",
29,hit_run,個人肇逃否,個人肇逃否,3,㉟,"1, 2",True,{},


,變數,2024欄,2025欄,PDF頁,表單欄,可核對印列碼,足以檢查碼外,特殊碼,限制
30,sight,道路障礙2,道路障礙2,1,⑪(2),"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13",False,{},同列1–7與8–13，不能逕認所有碼同一版本
31,cause,肇因碼-個別,肇因碼-個別,4,初步分析研判索引,"1, 2, 4, 5, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 27, 28, 29, 30, 31, 32, 33, 34, 35, ...",False,"{44: '尚未發現肇事因素', 67: '尚未發現肇事因素', 118: '跡證不足無法釐清', 119: '肇逃未查獲無法查明'}",可辨識印列碼集；版本／發布對應未認證，碼外先列未證實，不當官方無效
32,cause_main,肇因碼-主要,不存在,4,初步分析研判索引,"1, 2, 4, 5, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 27, 28, 29, 30, 31, 32, 33, 34, 35, ...",False,{},2024全缺失、2025無此欄；不可補0
33,occupation,職業,職業,3,㊱,"14, 15, 16, 24, 25, 26",False,{},PDF僅部分職業碼；21／22等大量未印列值不能逕判錯誤


**表 9：主要與補充變數逐碼定義**

... (90 outputs left out)

**表 11：cause_main 兩年實際碼分布（逐年分母，不檢定差異）**

,year,碼,原始碼定義,候選筆數,占全部候選%,占本欄非缺失%,狀態
0,2024,NA（原始缺失）,原始缺失,22368,100.0,<NA>,缺失


**表 11：occupation 兩年實際碼分布（逐年分母，不檢定差異）**

,year,碼,原始碼定義,候選筆數,占全部候選%,占本欄非缺失%,狀態
0,2024,7.0,本次碼表未列／定義待補,8,0.035765,0.036145,碼外／未證實
1,2024,14.0,小學生,18,0.080472,0.081327,印列碼
2,2024,15.0,國中生,20,0.089413,0.090363,印列碼
3,2024,16.0,高中職生,30,0.134120,0.135544,印列碼
4,2024,21.0,本次碼表未列／定義待補,16649,74.432225,75.222518,碼外／未證實
5,2024,22.0,本次碼表未列／定義待補,4885,21.839235,22.071116,碼外／未證實
6,2024,23.0,本次碼表未列／定義待補,40,0.178827,0.180726,碼外／未證實
7,2024,25.0,大學生,118,0.527539,0.533141,印列碼
8,2024,26.0,勞工人員,340,1.520029,1.536168,印列碼
9,2024,27.0,本次碼表未列／定義待補,12,0.053648,0.054218,碼外／未證實


## 4. Missing, Special and Invalid Values
所有比例分母是該年度第一當事人候選樣本；年度專屬欄在另一年是「不存在」，不計成原始缺失。
對可完整核對的類別碼，分開列缺失／特殊／碼外；部分碼表的「可核對碼外數」保留 NA。
年齡、速限等極端值只作標記，不宣稱必然錯誤，也不把任一欄缺失作整筆刪除條件。
時間有效性同時檢查時分整數與範圍；日期檢查曆法及與來源年度一致。
九個特徵各有自己的有效分母，週末所需日期和時段所需時分分開判定。


In [6]:
VALID={};DATES={};TIMES={};NUM={};COORD_TEXT={}
for year,frame in candidates.items():
    n={column:numeric(frame[column]) for column in frame.select_dtypes(include='number').columns}
    for column in ['發生年度','發生月','發生日','發生時-Hours','發生分','受傷人數','死亡人數','2-30日死亡人數','年齡','速限-速度限制','座標-X','座標-Y']:
        n[column]=numeric(frame[column])
    date=pd.to_datetime({'year':n['發生年度']+1911,'month':n['發生月'],'day':n['發生日']},errors='coerce')
    date_ok=(date.notna() & n['發生年度'].eq(year-1911)
             & nonnegative_integer(frame['發生年度']) & nonnegative_integer(frame['發生月'])
             & nonnegative_integer(frame['發生日']) & n['發生月'].between(1,12) & n['發生日'].between(1,31))
    time_ok=(nonnegative_integer(frame['發生時-Hours']) & n['發生時-Hours'].between(0,23)
             & nonnegative_integer(frame['發生分']) & n['發生分'].between(0,59))
    v={'date':date_ok.fillna(False),'time':time_ok.fillna(False),
       'datetime':(date_ok & time_ok).fillna(False),
       'type':frame['事故類型及型態'].isin(BOOKS['accident_type']['labels']),
       'position':frame['事故位置'].isin(BOOKS['position']['labels']),
       'injuries':nonnegative_integer(frame['受傷人數']),
       'fatalities_24h':nonnegative_integer(frame['死亡人數']),
       'fatalities_2_30d':nonnegative_integer(frame['2-30日死亡人數']),
       'vehicle':frame['車種'].isin(BOOKS['vehicle']['labels']),
       'sex_all_codes':frame['性別'].isin(BOOKS['sex']['labels']),
       'sex_male_female':frame['性別'].isin([1,2])}
    v['casualties_all_known']=v['injuries'] & v['fatalities_24h'] & v['fatalities_2_30d']
    raw_coordinate_text=pd.read_csv(project_file(INPUTS[year]),encoding='utf-8-sig',
             usecols=['座標-X','座標-Y'],dtype='string',keep_default_na=False)
    lexical=raw_coordinate_text.loc[frame.index]
    parsed=lexical.apply(pd.to_numeric,errors='coerce')
    coord_ok=parsed.notna().all(axis=1)&np.isfinite(parsed).all(axis=1)
    v['coordinates_parseable']=coord_ok.fillna(False)
    v['coordinates_global_degree_range']=(coord_ok & parsed['座標-X'].between(-180,180) & parsed['座標-Y'].between(-90,90)).fillna(False)
    DATES[year]=date;TIMES[year]=60*n['發生時-Hours']+n['發生分'];VALID[year]=v;NUM[year]=n;COORD_TEXT[year]=lexical

quality=[]
for year,frame in candidates.items():
    for column in frame:
        s=frame[column]
        bk=next((b for b in BOOKS.values() if b['columns'][year]==column),None)
        special=int(s.isin(bk['special']).sum()) if bk else pd.NA
        structural=pd.NA
        if column=='年齡':special=int((NUM[year][column].lt(0)|NUM[year][column].eq(0)|NUM[year][column].gt(100)).sum())
        if column=='速限-速度限制':special=int(NUM[year][column].isin([0,502]).sum())
        if column in ['受傷人數','死亡人數','2-30日死亡人數']:
            structural=int((s.notna() & ~nonnegative_integer(s)).sum())
        if column in ['座標-X','座標-Y']:
            structural=int((s.notna() & (~np.isfinite(numeric(s))|numeric(s).isna())).sum())
        outside=int((s.notna()&~s.isin(bk['labels'])).sum()) if bk else pd.NA
        quality.append({'year':year,'原始欄':column,'候選N':len(frame),'缺失筆數':int(s.isna().sum()),
             '缺失%':100*s.isna().sum()/len(frame),'印列特殊／預先標記數':special,
             '本次印列碼外數':outside,'可核對碼外數':outside if bk and bk['complete'] else pd.NA,
             '數值非負整數或解析結構異常':structural,
             '判讀限制':bk['caveat'] if bk else '無有限碼表；不能把未定義範圍判成官方錯誤'})
quality_table=pd.DataFrame(quality)
show('表 12：兩年第一當事人全欄缺失、特殊與碼外診斷',quality_table,block=15)
for year,frame in candidates.items():
    show(f'表 13：{year} 全數值原欄分布；類別碼分位數不作實質解釋',
         frame.select_dtypes(include='number').describe(percentiles=[.01,.25,.5,.75,.99]).T,block=20)
anomalies=[]
for year,frame in candidates.items():
    age=NUM[year]['年齡'];speed=NUM[year]['速限-速度限制']
    anomalies.append({'year':year,'speed_0':int(speed.eq(0).sum()),'speed_502':int(speed.eq(502).sum()),
        'speed_missing':int(frame['速限-速度限制'].isna().sum()),'age_negative':int(age.lt(0).sum()),
        'age_0':int(age.eq(0).sum()),'age_above_100':int(age.gt(100).sum()),'age_missing':int(age.isna().sum()),
        'invalid_calendar_or_source_year':int((~VALID[year]['date']).sum()),'invalid_hour_or_minute':int((~VALID[year]['time']).sum()),
        'vehicle_missing':int(frame['車種'].isna().sum()),'sex_missing':int(frame['性別'].isna().sum()),
        'sex_special_3_4_5':int(frame['性別'].isin([3,4,5]).sum())})
anomaly_table=pd.DataFrame(anomalies)
show('表 14：特別檢查欄位；預設標記不等於錯誤',anomaly_table)


**表 12：兩年第一當事人全欄缺失、特殊與碼外診斷**

,year,原始欄,候選N,缺失筆數,缺失%,印列特殊／預先標記數,本次印列碼外數,可核對碼外數,數值非負整數或解析結構異常,判讀限制
0,2024,發生年度,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
1,2024,發生月,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
2,2024,發生日,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
3,2024,發生時-Hours,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
4,2024,發生分,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
5,2024,區序,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
6,2024,肇事地點,22368,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
7,2024,死亡人數,22368,9,0.040236,<NA>,<NA>,<NA>,0,無有限碼表；不能把未定義範圍判成官方錯誤
8,2024,2-30日死亡人數,22368,9,0.040236,<NA>,<NA>,<NA>,0,無有限碼表；不能把未定義範圍判成官方錯誤
9,2024,受傷人數,22368,1,0.004471,<NA>,<NA>,<NA>,0,無有限碼表；不能把未定義範圍判成官方錯誤


,year,原始欄,候選N,缺失筆數,缺失%,印列特殊／預先標記數,本次印列碼外數,可核對碼外數,數值非負整數或解析結構異常,判讀限制
15,2024,速限-速度限制,22368,1,0.004471,329,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
16,2024,道路型態,22368,9,0.040236,0,0,0,<NA>,
17,2024,事故位置,22368,9,0.040236,0,0,0,<NA>,
18,2024,路面狀況1,22368,9,0.040236,0,0,0,<NA>,
19,2024,路面狀況2,22368,9,0.040236,0,0,0,<NA>,
20,2024,路面狀況3,22368,9,0.040236,0,0,0,<NA>,
21,2024,道路障礙1,22368,9,0.040236,0,0,0,<NA>,
22,2024,道路障礙2,22368,9,0.040236,0,0,<NA>,<NA>,同列1–7與8–13，不能逕認所有碼同一版本
23,2024,號誌1,22368,9,0.040236,12407,0,0,<NA>,
24,2024,號誌2,22368,9,0.040236,12407,0,0,<NA>,


,year,原始欄,候選N,缺失筆數,缺失%,印列特殊／預先標記數,本次印列碼外數,可核對碼外數,數值非負整數或解析結構異常,判讀限制
30,2024,性別,22368,0,0.000000,292,0,0,<NA>,
31,2024,年齡,22368,6,0.026824,234,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
32,2024,受傷程度,22368,227,1.014843,486,0,0,<NA>,
33,2024,主要傷處,22368,227,1.014843,12038,0,0,<NA>,
34,2024,保護裝置,22368,232,1.037196,5469,0,0,<NA>,
35,2024,行動電話,22368,230,1.028255,6201,0,0,<NA>,
36,2024,車輛用途,22368,225,1.005901,261,0,0,<NA>,
37,2024,當事者行動狀態,22368,229,1.023784,170,0,0,<NA>,
38,2024,駕駛資格情形,22368,224,1.001431,1456,0,0,<NA>,
39,2024,駕駛執照種類,22368,243,1.086373,2074,0,0,<NA>,


,year,原始欄,候選N,缺失筆數,缺失%,印列特殊／預先標記數,本次印列碼外數,可核對碼外數,數值非負整數或解析結構異常,判讀限制
45,2024,個人肇逃否,22368,37,0.165415,0,0,0,<NA>,
46,2024,職業,22368,235,1.050608,0,21607,<NA>,<NA>,PDF僅部分職業碼；21／22等大量未印列值不能逕判錯誤
47,2024,座標-X,22368,1,0.004471,<NA>,<NA>,<NA>,0,無有限碼表；不能把未定義範圍判成官方錯誤
48,2024,座標-Y,22368,1,0.004471,<NA>,<NA>,<NA>,0,無有限碼表；不能把未定義範圍判成官方錯誤
49,2025,發生年度,22762,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
50,2025,發生月,22762,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
51,2025,發生日,22762,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
52,2025,發生時-Hours,22762,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
53,2025,發生分,22762,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
54,2025,區序,22762,0,0.000000,<NA>,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤


,year,原始欄,候選N,缺失筆數,缺失%,印列特殊／預先標記數,本次印列碼外數,可核對碼外數,數值非負整數或解析結構異常,判讀限制
60,2025,車種,22762,157,0.689746,257,0,0,<NA>,
61,2025,天候,22762,2,0.008787,0,0,0,<NA>,
62,2025,光線,22762,12,0.052719,0,0,<NA>,<NA>,表單同列1–4與5–7；兩年欄位含義及代碼版本等價尚未證實，僅核對印列值
63,2025,道路類別,22762,12,0.052719,0,0,0,<NA>,
64,2025,速限-速度限制,22762,4,0.017573,315,<NA>,<NA>,<NA>,無有限碼表；不能把未定義範圍判成官方錯誤
65,2025,道路型態,22762,12,0.052719,0,0,0,<NA>,
66,2025,事故位置,22762,12,0.052719,0,0,0,<NA>,
67,2025,路面狀況1,22762,12,0.052719,0,0,0,<NA>,
68,2025,路面狀況2,22762,12,0.052719,0,0,0,<NA>,
69,2025,路面狀況3,22762,12,0.052719,0,0,0,<NA>,


... (8 outputs left out)

,count,mean,std,min,1%,25%,50%,75%,99%,max
號誌1,22750.0,2.908000,1.265954,1.0,1.0,2.0,4.0,4.0,4.0,4.0
號誌2,22750.0,2.650462,1.490581,1.0,1.0,1.0,4.0,4.0,4.0,4.0
車道劃分-分向,22750.0,5.371604,3.645113,1.0,1.0,1.0,5.0,9.0,10.0,10.0
車道劃分-分道1,22750.0,4.192571,0.759060,1.0,2.0,4.0,4.0,5.0,5.0,5.0
車道劃分-分道2,22750.0,4.813275,0.647674,1.0,1.0,5.0,5.0,5.0,5.0,5.0
車道劃分-分道3,22750.0,1.824967,0.380004,1.0,1.0,2.0,2.0,2.0,2.0,2.0
事故類型及型態,22750.0,14.486549,3.937048,1.0,2.0,13.0,16.0,17.0,23.0,38.0
性別,22760.0,1.303427,0.519852,1.0,1.0,1.0,1.0,2.0,3.0,4.0
年齡,22750.0,44.376747,17.240335,0.0,7.0,30.0,44.0,58.0,80.0,113.0
受傷程度,22547.0,2.561272,0.537270,1.0,2.0,2.0,3.0,3.0,4.0,5.0


,count,mean,std,min,1%,25%,50%,75%,99%,max
個人肇逃否,22740.0,1.038874,0.193299,1.00000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000
職業,22551.0,21.346104,0.952198,14.00000,21.000000,21.000000,21.000000,21.000000,26.000000,29.000000
座標-X,22760.0,121.540491,0.030742,121.46118,121.484224,121.516802,121.536655,121.562316,121.616421,121.652018
座標-Y,22760.0,25.054330,0.032454,24.96593,24.985190,25.034563,25.049958,25.068917,25.138260,25.183295


**表 14：特別檢查欄位；預設標記不等於錯誤**

,year,speed_0,speed_502,speed_missing,age_negative,age_0,age_above_100,age_missing,invalid_calendar_or_source_year,invalid_hour_or_minute,vehicle_missing,sex_missing,sex_special_3_4_5
0,2024,328,1,1,1,222,11,6,0,0,91,0,292
1,2025,315,0,4,0,217,18,12,0,0,157,2,258


## 5. Coordinate Validation and CRS Evidence
查閱上述官方事故資料頁、政府資料平臺及官方表單，並搜尋本資料集與 WGS84／座標系統／EPSG 的相關說明；
本次未找到足以把本機 X／Y 直接認證為 EPSG:4326 的生產者證據。
以下區分缺失、原始文字非數值、非有限數值與數值範圍。全球角度界限只作格式診斷，不是臺北市邊界驗證。
本次沒有讀里界、指定CRS、投影、geocoding或空間歸里。


In [7]:
EVIDENCE = [{'evidence_id': 'S1', '來源': '臺北市死傷交通事故資料官方頁', 'URL': 'https://data.taipei/dataset/detail?id=2f238b4f-1b27-4085-93e9-d684ef0e2735', '查閱日': '2026-11-26', '位置': '詮釋資料／主要欄位／備註', '查得內容': '列X、Y與24小時事故分類，連結112.7.1後表單；未明示此批座標datum／EPSG', '證據範圍': '支持原始欄名及表單來源；不能確認本機CRS或下載版本'}, {'evidence_id': 'S2', '來源': '政府資料開放平臺 臺北市死傷交通事故資料', 'URL': 'https://data.gov.tw/dataset/130110', '查閱日': '2026-11-26', '位置': '主要欄位／提供機關／備註', '查得內容': '同機關資料集；備註稱X、Y座標點位，未取得明確CRS，欄位清單也非本機完整schema', '證據範圍': '官方交叉查證仍未解決座標基準；不把其他事故產品的CRS套用'}, {'evidence_id': 'S3', '來源': '官方112.7.1後道路交通事故調查報告表', 'URL': 'https://www-ws.gov.taipei/001/Upload/467/relfile/32284/5040008/7ddb52e9-ebae-4834-b41a-5f41d5074279.pdf', '查閱日': '2026-11-26', '位置': 'p.1／p.3／p.4；本機PDF亦同適用標示', '查得內容': '列事故位置、型態、死傷欄及當事人代碼；p.1死亡欄標24小時內，未定義CSV座標datum', '證據範圍': '表單支持死亡欄24小時候選解釋；CSV欄位生成／2–30日彙整及跨年對應待官方說明'}, {'evidence_id': 'S4', '來源': '本機CSV數值及固定hash', 'URL': '../raw/taipei_2024_2025_crashes_raw_documentation.ipynb', '查閱日': '2026-11-26', '位置': '表15–16實際執行輸出', '查得內容': 'X約121、Y約25，與經緯度形式相容；非百萬／十萬公尺形式', '證據範圍': '數值只能支持角度座標候選，不能區辨WGS84／TWD97地理座標等datum'}]
show('表 15：座標基準與死亡欄位來源證據及限制',pd.DataFrame(EVIDENCE),block=5)
coordinate_rows=[]
for year,frame in candidates.items():
    lex=COORD_TEXT[year];parsed=lex.apply(pd.to_numeric,errors='coerce')
    for column in ['座標-X','座標-Y']:
        text_nonempty=lex[column].str.strip().ne('')
        values=parsed[column]
        quantiles=values.quantile([0,.01,.25,.5,.75,.99,1])
        coordinate_rows.append({'year':year,'column':column,'候選N':len(frame),'原始文字空白':int((~text_nonempty).sum()),
           '非空非數值':int((text_nonempty&values.isna()).sum()),'非有限數值':int((values.notna()&~np.isfinite(values)).sum()),
           **{f'q{q:g}':float(value) for q,value in quantiles.items()}})
    parseable=VALID[year]['coordinates_parseable']
    print({'year':year,'missing_or_unparseable_coordinate_pairs':int((~parseable).sum()),
       'outside_global_degree_range':int((parseable&~VALID[year]['coordinates_global_degree_range']).sum()),
       'zero_coordinate_pairs':int((parseable&(parsed['座標-X'].eq(0)|parsed['座標-Y'].eq(0))).sum()),
       'native_xy_may_look_swapped':int((parseable&parsed['座標-X'].between(-90,90)&parsed['座標-Y'].abs().gt(90)).sum()),
       'administrative_boundary_containment':'not tested; no spatial join'})
show('表 16：兩年X／Y缺失、非數值及主要分位數',pd.DataFrame(coordinate_rows))
from pyproj import CRS
show('表 17：候選CRS登錄名稱；只查metadata，不指派給資料',pd.DataFrame([
    {'候選EPSG':epsg,'登錄名稱':CRS.from_epsg(epsg).name,'與數值形式':'角度座標可相容','本批證據':'尚未確認'}
    for epsg in [4326,3824]]))
print({'coordinate_CRS_status':'UNCONFIRMED','working_candidate':'WGS84 / EPSG:4326 (inference only)',
       'alternative_not_excluded':'TWD97 geographic / EPSG:3824 or other geographic datum',
       'rationale':'X/Y resemble longitude/latitude; numerical plausibility cannot identify datum',
       'no_set_crs_or_transform':True})


**表 15：座標基準與死亡欄位來源證據及限制**

,evidence_id,來源,URL,查閱日,位置,查得內容,證據範圍
0,S1,臺北市死傷交通事故資料官方頁,https://data.taipei/dataset/detail?id=2f238b4f-1b27-4085-93e9-d684ef0e2735,2026-11-26,詮釋資料／主要欄位／備註,列X、Y與24小時事故分類，連結112.7.1後表單；未明示此批座標datum／EPSG,支持原始欄名及表單來源；不能確認本機CRS或下載版本
1,S2,政府資料開放平臺 臺北市死傷交通事故資料,https://data.gov.tw/dataset/130110,2026-11-26,主要欄位／提供機關／備註,同機關資料集；備註稱X、Y座標點位，未取得明確CRS，欄位清單也非本機完整schema,官方交叉查證仍未解決座標基準；不把其他事故產品的CRS套用
2,S3,官方112.7.1後道路交通事故調查報告表,https://www-ws.gov.taipei/001/Upload/467/relfile/32284/5040008/7ddb52e9-ebae-4834-b41a-5f41d5074279.pdf,2026-11-26,p.1／p.3／p.4；本機PDF亦同適用標示,列事故位置、型態、死傷欄及當事人代碼；p.1死亡欄標24小時內，未定義CSV座標datum,表單支持死亡欄24小時候選解釋；CSV欄位生成／2–30日彙整及跨年對應待官方說明
3,S4,本機CSV數值及固定hash,../raw/taipei_2024_2025_crashes_raw_documentation.ipynb,2026-11-26,表15–16實際執行輸出,X約121、Y約25，與經緯度形式相容；非百萬／十萬公尺形式,數值只能支持角度座標候選，不能區辨WGS84／TWD97地理座標等datum


{'year': 2024, 'missing_or_unparseable_coordinate_pairs': 1, 'outside_global_degree_range': 0, 'zero_coordinate_pairs': 0, 'native_xy_may_look_swapped': 0, 'administrative_boundary_containment': 'not tested; no spatial join'}
{'year': 2025, 'missing_or_unparseable_coordinate_pairs': 2, 'outside_global_degree_range': 0, 'zero_coordinate_pairs': 0, 'native_xy_may_look_swapped': 0, 'administrative_boundary_containment': 'not tested; no spatial join'}


**表 16：兩年X／Y缺失、非數值及主要分位數**

,year,column,候選N,原始文字空白,非空非數值,非有限數值,q0,q0.01,q0.25,q0.5,q0.75,q0.99,q1
0,2024,座標-X,22368,1,0,0,121.461189,121.483760,121.517018,121.536938,121.563038,121.616357,121.643370
1,2024,座標-Y,22368,1,0,0,24.966712,24.984301,25.033735,25.049174,25.068656,25.136971,25.188246
2,2025,座標-X,22762,2,0,0,121.461180,121.484224,121.516802,121.536655,121.562316,121.616421,121.652018
3,2025,座標-Y,22762,2,0,0,24.965930,24.985190,25.034563,25.049958,25.068917,25.138260,25.183295


**表 17：候選CRS登錄名稱；只查metadata，不指派給資料**

,候選EPSG,登錄名稱,與數值形式,本批證據
0,4326,WGS 84,角度座標可相容,尚未確認
1,3824,TWD97,角度座標可相容,尚未確認


{'coordinate_CRS_status': 'UNCONFIRMED', 'working_candidate': 'WGS84 / EPSG:4326 (inference only)', 'alternative_not_excluded': 'TWD97 geographic / EPSG:3824 or other geographic datum', 'rationale': 'X/Y resemble longitude/latitude; numerical plausibility cannot identify datum', 'no_set_crs_or_transform': True}


**座標判斷：** 已確認兩年 X／Y 的實際範圍、可解析性及三筆候選缺座標；
X 為經度、Y 為緯度是合理候選，WGS84／EPSG:4326 可作待驗證工作假設，不能寫成已確認CRS。
僅靠範圍不能排除 TWD97 地理座標（EPSG:3824）等基準，也未證明所有點在臺北行政範圍內。

後續方案：先取得交通大隊對113／114資源的欄位規格或書面基準說明（本次未聯絡他人），
再以有獨立可靠座標的已知地點抽查軸序、偏移及不同候選datum；必要時經學生授權比較候選投影結果。
幾何「看起來吻合」只能當輔助，不能取代來源基準證據。未確認前不做正式事故歸里。


## 6. Dataset A — Formal Candidate Operational Definitions
下列九項是依學生指示正式記錄的「候選」操作型定義，不是已決定的最終分群輸入。
各欄有效分母獨立；分子條件只在有效集合內計算。早晚尖峰是預設時段，不代表實際車流量；
人與車分類不代表完整行人涉入比例，多人受傷不代表傷勢嚴重。
未來每組需保留 numerator、valid_denominator、missing_or_unusable_count、coverage；
低樣本、零事故及分母0不自動刪里或填0，不直接平均兩年比例。


In [8]:
FEATURE_DEFINITIONS = [{'變數': 'morning_peak_ratio', '中文定義': '早晨預設時段比例', '原始欄位': '發生時-Hours、發生分', '分子條件': '07:00 ≤ t < 09:00；包含07:00、排除09:00', '有效分母條件': '時、分為可解析整數且0≤h≤23、0≤m≤59', '缺失與特殊值': '僅此特徵排除無效時分；不要求其他欄完整', '跨年一致性': '兩年同欄；同端點', '候選敏感度': '平日限定：另要求日期有效且星期一至五', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of eligible crashes occurring at 07:00 inclusive to 09:00 exclusive.'}, {'變數': 'evening_peak_ratio', '中文定義': '傍晚預設時段比例', '原始欄位': '發生時-Hours、發生分', '分子條件': '17:00 ≤ t < 19:00；包含17:00、排除19:00', '有效分母條件': '有效時分', '缺失與特殊值': '同上，不因其他欄缺失刪除', '跨年一致性': '兩年同欄；同端點', '候選敏感度': '平日限定；替代時段另待學生確認', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of eligible crashes occurring at 17:00 inclusive to 19:00 exclusive.'}, {'變數': 'late_night_ratio', '中文定義': '深夜預設時段比例', '原始欄位': '發生時-Hours、發生分', '分子條件': '22:00 ≤ t < 24:00 或 00:00 ≤ t < 06:00；排除06:00', '有效分母條件': '有效時分', '缺失與特殊值': '只影響時段分母；不依照明代碼推定時間', '跨年一致性': '兩年同欄；按事故當日日曆時間', '候選敏感度': '天文日落／日出或其他端點候選，須另授權實作', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share occurring at or after 22:00 or before 06:00; 06:00 is excluded.'}, {'變數': 'weekend_ratio', '中文定義': '週末事故比例', '原始欄位': '發生年度、發生月、發生日', '分子條件': '有效日期之星期六或日', '有效分母條件': '有效曆法日期且年份與來源年相符；不要求時分', '缺失與特殊值': '日期無法判定則排除此特徵分母；非所有國定假日', '跨年一致性': '民國年+1911後使用相同曆法', '候選敏感度': '國定假日另需正式行事曆，不在此資料假造', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of crashes with a valid date occurring on Saturday or Sunday.'}, {'變數': 'intersection_ratio', '中文定義': '路口及附近事故比例', '原始欄位': '事故位置', '分子條件': '代碼∈{1,2,3,4}', '有效分母條件': '事故位置屬PDF p.1⑨可核對碼集', '缺失與特殊值': '缺失／碼外不當非路口；22其他仍是有效分類', '跨年一致性': '兩年同名；主碼1–4可核對', '候選敏感度': '嚴格路口內：分子只取1；分母仍有效位置', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of crashes with valid position codes in categories 1 through 4.'}, {'變數': 'rear_end_ratio', '中文定義': '追撞事故比例', '原始欄位': '事故類型及型態', '分子條件': '代碼=13', '有效分母條件': '事故類型屬PDF p.1⑮可核對碼集', '缺失與特殊值': '缺失／碼外不當未追撞；其他有效碼保留分母', '跨年一致性': '兩年同名；p.2定義同向前後車碰撞', '候選敏感度': '排除寬泛其他碼會改變目標母體，須另審查', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of crashes with valid accident-type codes classified as type 13.'}, {'變數': 'side_collision_ratio', '中文定義': '側撞事故比例', '原始欄位': '事故類型及型態', '分子條件': '代碼=16', '有效分母條件': '有效事故型態', '缺失與特殊值': '缺失／碼外只影響型態分母', '跨年一致性': 'p.2：轉向過程與直行車碰撞；不含變換車道擦撞', '候選敏感度': '不與11／12擦撞合併；替代分類另審查', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of crashes with valid accident-type codes classified as type 16.'}, {'變數': 'pedestrian_collision_ratio', '中文定義': '表單人與車事故比例', '原始欄位': '事故類型及型態', '分子條件': '代碼∈{1,…,9}', '有效分母條件': '有效事故型態', '缺失與特殊值': '9人與車其他仍在分子；缺失不當沒有行人', '跨年一致性': 'p.1⑮人與車類別，兩年同名', '候選敏感度': '若研究全部行人涉入須其他證據，不能由第一當事人車種取代', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share classified as person-versus-vehicle types 1 through 9; not all pedestrian involvement.'}, {'變數': 'multi_injury_ratio', '中文定義': '至少兩人受傷事故比例', '原始欄位': '受傷人數', '分子條件': '事故受傷人數≥2', '有效分母條件': '原始受傷人數為已知、有限、非負整數', '缺失與特殊值': '缺失不是0；不以第一當事人受傷程度代填', '跨年一致性': '同名事故總量欄，缺值分列', '候選敏感度': '移出Dataset A、只作Dataset C剖析；不代表重傷程度', '公式': 'ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA', '適用範圍': '未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷', '共同限制': '非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查', 'English definition': 'Share of crashes with a known nonnegative integer injury count of at least two.'}]
feature_table=pd.DataFrame(FEATURE_DEFINITIONS)
for columns in [['變數','中文定義','English definition','原始欄位','分子條件','有效分母條件'],
                ['變數','缺失與特殊值','跨年一致性','候選敏感度'],['變數','公式','適用範圍','共同限制']]:
    show('表 18：Dataset A完整候選操作定義',feature_table[columns],block=9)
A_MASKS={};feature_coverage=[]
for year,frame in candidates.items():
    v=VALID[year];minutes=TIMES[year];date=DATES[year]
    definitions={
      'morning_peak_ratio':(v['time'],minutes.ge(420)&minutes.lt(540)),
      'evening_peak_ratio':(v['time'],minutes.ge(1020)&minutes.lt(1140)),
      'late_night_ratio':(v['time'],minutes.ge(1320)|minutes.lt(360)),
      'weekend_ratio':(v['date'],date.dt.day_of_week.isin([5,6])),
      'intersection_ratio':(v['position'],frame['事故位置'].isin([1,2,3,4])),
      'rear_end_ratio':(v['type'],frame['事故類型及型態'].eq(13)),
      'side_collision_ratio':(v['type'],frame['事故類型及型態'].eq(16)),
      'pedestrian_collision_ratio':(v['type'],frame['事故類型及型態'].isin(range(1,10))),
      'multi_injury_ratio':(v['injuries'],NUM[year]['受傷人數'].ge(2)),
    }
    A_MASKS[year]=definitions
    for variable,(eligible,condition) in definitions.items():
        denominator=int(eligible.sum());numerator=int((eligible & condition.fillna(False)).sum())
        assert 0<=numerator<=denominator<=len(frame)
        feature_coverage.append({'year':year,'候選變數':variable,'候選N':len(frame),
            '有效分母':denominator,'有效分子':numerator,'不可判定':len(frame)-denominator,
            '涵蓋率%':100*denominator/len(frame),'逐年全市診斷比例':numerator/denominator if denominator else pd.NA})
feature_coverage_table=pd.DataFrame(feature_coverage)
show('表 19：九個特徵的實際分子、有效分母及涵蓋率；不是里層級特徵表',feature_coverage_table,block=18)
supplement=[]
for year,frame in candidates.items():
    age=NUM[year]['年齡'];speed=NUM[year]['速限-速度限制'];v=VALID[year]
    for variable,column,eligible,note in [
      ('第一當事人年齡','年齡',nonnegative_integer(frame['年齡']),'此處只檢查非負整數；0／>100與自然人資格仍待審查，不自動判無效'),
      ('第一當事人性別：全部表單碼','性別',v['sex_all_codes'],'含3–5特殊狀態，不能全當男／女'),
      ('第一當事人性別：男／女候選','性別',v['sex_male_female'],'3–5與缺失退出此補充性別分母，事故本身仍保留'),
      ('第一當事人車種','車種',v['vehicle'],'只能描述第一當事人，不是全部涉入運具'),
      ('速限原數值','速限-速度限制',nonnegative_integer(frame['速限-速度限制']),'0／502仍為數值但語意待補；下列規則只作敏感度'),
      ('天候','天候',frame['天候'].isin(BOOKS['weather']['labels']),'可建立類別補充，不估車流暴露'),
      ('道路型態','道路型態',frame['道路型態'].isin(BOOKS['road_shape']['labels']),'只描述事故道路情境，不代表全里道路供給'),
    ]:
        supplement.append({'year':year,'補充變數':variable,'原始非缺失':int(frame[column].notna().sum()),
                           '結構／碼表可判定N':int(eligible.sum()),'限制':note})
show('表 20：補充變數可建立性；未決定分組界線',pd.DataFrame(supplement),block=14)


**表 18：Dataset A完整候選操作定義**

,變數,中文定義,English definition,原始欄位,分子條件,有效分母條件
0,morning_peak_ratio,早晨預設時段比例,Share of eligible crashes occurring at 07:00 inclusive to 09:00 exclusive.,發生時-Hours、發生分,07:00 ≤ t < 09:00；包含07:00、排除09:00,時、分為可解析整數且0≤h≤23、0≤m≤59
1,evening_peak_ratio,傍晚預設時段比例,Share of eligible crashes occurring at 17:00 inclusive to 19:00 exclusive.,發生時-Hours、發生分,17:00 ≤ t < 19:00；包含17:00、排除19:00,有效時分
2,late_night_ratio,深夜預設時段比例,Share occurring at or after 22:00 or before 06:00; 06:00 is excluded.,發生時-Hours、發生分,22:00 ≤ t < 24:00 或 00:00 ≤ t < 06:00；排除06:00,有效時分
3,weekend_ratio,週末事故比例,Share of crashes with a valid date occurring on Saturday or Sunday.,發生年度、發生月、發生日,有效日期之星期六或日,有效曆法日期且年份與來源年相符；不要求時分
4,intersection_ratio,路口及附近事故比例,Share of crashes with valid position codes in categories 1 through 4.,事故位置,"代碼∈{1,2,3,4}",事故位置屬PDF p.1⑨可核對碼集
5,rear_end_ratio,追撞事故比例,Share of crashes with valid accident-type codes classified as type 13.,事故類型及型態,代碼=13,事故類型屬PDF p.1⑮可核對碼集
6,side_collision_ratio,側撞事故比例,Share of crashes with valid accident-type codes classified as type 16.,事故類型及型態,代碼=16,有效事故型態
7,pedestrian_collision_ratio,表單人與車事故比例,Share classified as person-versus-vehicle types 1 through 9; not all pedestrian involvement.,事故類型及型態,"代碼∈{1,…,9}",有效事故型態
8,multi_injury_ratio,至少兩人受傷事故比例,Share of crashes with a known nonnegative integer injury count of at least two.,受傷人數,事故受傷人數≥2,原始受傷人數為已知、有限、非負整數


**表 18：Dataset A完整候選操作定義**

,變數,缺失與特殊值,跨年一致性,候選敏感度
0,morning_peak_ratio,僅此特徵排除無效時分；不要求其他欄完整,兩年同欄；同端點,平日限定：另要求日期有效且星期一至五
1,evening_peak_ratio,同上，不因其他欄缺失刪除,兩年同欄；同端點,平日限定；替代時段另待學生確認
2,late_night_ratio,只影響時段分母；不依照明代碼推定時間,兩年同欄；按事故當日日曆時間,天文日落／日出或其他端點候選，須另授權實作
3,weekend_ratio,日期無法判定則排除此特徵分母；非所有國定假日,民國年+1911後使用相同曆法,國定假日另需正式行事曆，不在此資料假造
4,intersection_ratio,缺失／碼外不當非路口；22其他仍是有效分類,兩年同名；主碼1–4可核對,嚴格路口內：分子只取1；分母仍有效位置
5,rear_end_ratio,缺失／碼外不當未追撞；其他有效碼保留分母,兩年同名；p.2定義同向前後車碰撞,排除寬泛其他碼會改變目標母體，須另審查
6,side_collision_ratio,缺失／碼外只影響型態分母,p.2：轉向過程與直行車碰撞；不含變換車道擦撞,不與11／12擦撞合併；替代分類另審查
7,pedestrian_collision_ratio,9人與車其他仍在分子；缺失不當沒有行人,p.1⑮人與車類別，兩年同名,若研究全部行人涉入須其他證據，不能由第一當事人車種取代
8,multi_injury_ratio,缺失不是0；不以第一當事人受傷程度代填,同名事故總量欄，缺值分列,移出Dataset A、只作Dataset C剖析；不代表重傷程度


**表 18：Dataset A完整候選操作定義**

,變數,公式,適用範圍,共同限制
0,morning_peak_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
1,evening_peak_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
2,late_night_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
3,weekend_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
4,intersection_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
5,rear_end_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
6,side_collision_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
7,pedestrian_collision_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查
8,multi_injury_ratio,ratio_g = 分組內有效且符合條件之候選事件數 / 分組內此特徵可判定事件數；分母0為NA,未來行政里×年，或合計兩年分子／分母後之里；本次僅逐年全市候選診斷,非暴露校正事故率；不代表當事人居民；未完成GIS與最終特徵審查


**表 19：九個特徵的實際分子、有效分母及涵蓋率；不是里層級特徵表**

,year,候選變數,候選N,有效分母,有效分子,不可判定,涵蓋率%,逐年全市診斷比例
0,2024,morning_peak_ratio,22368,22368,3323,0,100.000000,0.148560
1,2024,evening_peak_ratio,22368,22368,3332,0,100.000000,0.148963
2,2024,late_night_ratio,22368,22368,2119,0,100.000000,0.094734
3,2024,weekend_ratio,22368,22368,4988,0,100.000000,0.222997
4,2024,intersection_ratio,22368,22359,13556,9,99.959764,0.606288
5,2024,rear_end_ratio,22368,22359,3131,9,99.959764,0.140033
6,2024,side_collision_ratio,22368,22359,4483,9,99.959764,0.200501
7,2024,pedestrian_collision_ratio,22368,22359,1969,9,99.959764,0.088063
8,2024,multi_injury_ratio,22368,22367,6253,1,99.995529,0.279564
9,2025,morning_peak_ratio,22762,22762,3415,0,100.000000,0.150031


**表 20：補充變數可建立性；未決定分組界線**

,year,補充變數,原始非缺失,結構／碼表可判定N,限制
0,2024,第一當事人年齡,22362,22361,此處只檢查非負整數；0／>100與自然人資格仍待審查，不自動判無效
1,2024,第一當事人性別：全部表單碼,22368,22368,含3–5特殊狀態，不能全當男／女
2,2024,第一當事人性別：男／女候選,22368,22076,3–5與缺失退出此補充性別分母，事故本身仍保留
3,2024,第一當事人車種,22277,22277,只能描述第一當事人，不是全部涉入運具
4,2024,速限原數值,22367,22367,0／502仍為數值但語意待補；下列規則只作敏感度
5,2024,天候,22368,22368,可建立類別補充，不估車流暴露
6,2024,道路型態,22359,22359,只描述事故道路情境，不代表全里道路供給
7,2025,第一當事人年齡,22750,22750,此處只檢查非負整數；0／>100與自然人資格仍待審查，不自動判無效
8,2025,第一當事人性別：全部表單碼,22760,22760,含3–5特殊狀態，不能全當男／女
9,2025,第一當事人性別：男／女候選,22760,22502,3–5與缺失退出此補充性別分母，事故本身仍保留


## 7. Dataset C — Profiling Candidates
死亡欄24小時解釋有表單支持，但尚未取得本機 CSV 的明確發布映射；保留暫定名稱與限制。
以下不合併兩個死亡欄，也不把第一當事人受傷程度當作事故傷亡總數。
每平方公里事故數、低樣本標記與里年彙整延至里界／事故空間整合完成。

**新增待查問題：** 第一當事人程度碼5與2–30日死亡原欄之間，有2024年6筆、2025年4筆「個人碼為死亡但事件總量為0」。若兩欄為同事件同追蹤時點，這是單向一致性疑點；目前不能排除發布／更新口徑差異，不直接判定哪欄錯誤。不以程度碼填補或改寫總量，結構有效N也不等於語義已驗證。程度碼1對24小時死亡原欄及碼2對受傷人數未見同類已知零值疑點。


In [9]:
C_DEFINITIONS = [{'變數': 'annual_accident_count', '來源': '第一當事人研究用record_id', '分子／公式': '未來每里每年已歸屬候選事件的distinct record_id數，不按候選鍵自動去重', '有效分母／缺失': '計數無比例分母；須保留未歸里及來源涵蓋狀態', '狀態': '操作口徑已確認，最終年度里層級計數尚不可建立', '限制': '本次僅逐年候選N；不等於所有交通事故或暴露校正事故率'}, {'變數': 'total_injuries', '來源': '受傷人數', '分子／公式': 'sum(有效事故受傷人數)，全無有效值則NA', '有效分母／缺失': '同記有效N／缺失N／結構異常N；缺失不補0，不稱已觀測和為完整總數', '狀態': '總量欄候選；未來按里年彙整', '限制': '不以第一當事人受傷程度代替；與A多人受傷相關，不是獨立驗證'}, {'變數': 'total_fatalities_24h', '來源': '死亡人數', '分子／公式': '暫定sum(有效死亡人數)，全無有效值則NA', '有效分母／缺失': '非負整數結構檢查；記有效N、缺失與異常N', '狀態': '暫定英文名；PDF p.1支持24小時，但CSV發布映射尚缺', '限制': '未認證最終死亡統計口徑，不與2–30日死亡直接相加'}, {'變數': 'total_fatalities_2_30d', '來源': '2-30日死亡人數', '分子／公式': '暫定sum(有效原欄人數)，全無有效值則NA', '有效分母／缺失': '記有效N、缺失N與異常N；保留原始欄意', '狀態': '欄名與PDF個人代碼支持候選，事故追蹤彙整規格未取得', '限制': '個人2–30日程度碼不能代替事件總量，亦未認證可與另一死亡欄無重複相加'}, {'變數': 'fatal_accident_ratio', '來源': '死亡人數（24小時候選口徑）', '分子／公式': '有效死亡人數≥1的事件數 / 死亡人數已知有效事件數', '有效分母／缺失': '只用該死亡欄的有效分母；分母0為NA', '狀態': '暫定24小時版本，須學生確認且補發布映射', '限制': '不是死亡人數除以事故數；不合併2–30日，不從第一當事人傷亡程度推算'}, {'變數': 'injuries_per_crash', '來源': '受傷人數', '分子／公式': '有效事故受傷人數總和 / 有效受傷人數事件N', '有效分母／缺失': '用有效N而非含缺失的全部候選N；分母0為NA', '狀態': '可作條件於傷亡資料已知的平均；未建里年數值', '限制': '有缺失時不能當全體事故平均，需同報涵蓋率；與A傷亡特徵非完全獨立'}]
show('表 21：Dataset C候選操作定義與統計口徑限制',pd.DataFrame(C_DEFINITIONS),block=6)
casualty_diagnostics=[]
for year,frame in candidates.items():
    for key,column in [('injuries','受傷人數'),('fatalities_24h','死亡人數'),('fatalities_2_30d','2-30日死亡人數')]:
        eligible=VALID[year][key];s=NUM[year][column];valid_count=int(eligible.sum())
        observed_sum=s.loc[eligible].sum(min_count=1)
        casualty_diagnostics.append({'year':year,'原欄':column,'全部候選N':len(frame),'有效N':valid_count,
          '原始缺失N':int(frame[column].isna().sum()),'非缺失結構異常N':int((frame[column].notna()&~eligible).sum()),
          '已觀測有效人數和':observed_sum,'資料完整總數可否認證':'否，仍有缺失或發布口徑待補',
          '正值事件數':int((eligible&s.gt(0)).sum()),'每有效事件原欄平均':observed_sum/valid_count if valid_count else pd.NA})
show('表 22：傷亡原欄的有效N、缺失及已觀測和；三欄保持分列',pd.DataFrame(casualty_diagnostics),block=6)
for year,frame in candidates.items():
    print({'year':year,'死亡人數正值事件':int((VALID[year]['fatalities_24h']&NUM[year]['死亡人數'].gt(0)).sum()),
           '第一當事人程度碼1':int(frame['受傷程度'].eq(1).sum()),
           '2至30日原欄正值事件':int((VALID[year]['fatalities_2_30d']&NUM[year]['2-30日死亡人數'].gt(0)).sum()),
           '第一當事人程度碼5':int(frame['受傷程度'].eq(5).sum()),
           'interpretation':'different units; no substitution, no automatic reconciliation or mortality-field combination'})


# Independent annual-city diagnostic previews only; no village-year data exist.
C_values=[]
for year,frame in candidates.items():
    count=len(frame)
    C_values.append({'year':year,'候選變數':'annual_accident_count','逐年全市診斷值':count,
                     '有效事件N':count,'來源缺失N':0,'狀態':'未GIS，非最終里年事故數'})
    for variable,key,column in [
      ('total_injuries','injuries','受傷人數'),
      ('total_fatalities_24h','fatalities_24h','死亡人數'),
      ('total_fatalities_2_30d','fatalities_2_30d','2-30日死亡人數')]:
        valid=VALID[year][key];valid_n=int(valid.sum());s=NUM[year][column]
        C_values.append({'year':year,'候選變數':variable,'逐年全市診斷值':s.loc[valid].sum(min_count=1),
            '有效事件N':valid_n,'來源缺失N':int(frame[column].isna().sum()),
            '狀態':'已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定'})
    valid=VALID[year]['fatalities_24h'];valid_n=int(valid.sum())
    C_values.append({'year':year,'候選變數':'fatal_accident_ratio',
        '逐年全市診斷值':int((valid&NUM[year]['死亡人數'].gt(0)).sum())/valid_n if valid_n else pd.NA,
        '有效事件N':valid_n,'來源缺失N':int(frame['死亡人數'].isna().sum()),
        '狀態':'僅死亡人數原欄正值事件比例；24小時映射暫定，不合併2–30日'})
    valid=VALID[year]['injuries'];valid_n=int(valid.sum())
    C_values.append({'year':year,'候選變數':'injuries_per_crash',
        '逐年全市診斷值':NUM[year]['受傷人數'].loc[valid].sum(min_count=1)/valid_n if valid_n else pd.NA,
        '有效事件N':valid_n,'來源缺失N':int(frame['受傷人數'].isna().sum()),
        '狀態':'條件於受傷人數已知的事件平均，非全體事故完整平均'})
show('表 22B：六項C候選的逐年全市記憶體診斷；非正式年度資料集',pd.DataFrame(C_values),block=12)


casualty_consistency=[]
for year,frame in candidates.items():
    for degree,column in [(1,'死亡人數'),(5,'2-30日死亡人數'),(2,'受傷人數')]:
        selected=frame['受傷程度'].eq(degree)
        casualty_consistency.append({'year':year,'第一當事人程度碼':degree,'對應事故總量原欄':column,
          '程度碼符合N':int(selected.sum()),'事故總量已知正值':int((selected&numeric(frame[column]).ge(1)).sum()),
          '事故總量為0的待查N':int((selected&numeric(frame[column]).eq(0)).sum()),
          '事故總量缺失N':int((selected&frame[column].isna()).sum()),
          '判讀':'若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補'})
show('表 22C：個人程度碼與事件總量的單向條件式一致性；不要求兩種單位數值相等',pd.DataFrame(casualty_consistency),block=6)


**表 21：Dataset C候選操作定義與統計口徑限制**

,變數,來源,分子／公式,有效分母／缺失,狀態,限制
0,annual_accident_count,第一當事人研究用record_id,未來每里每年已歸屬候選事件的distinct record_id數，不按候選鍵自動去重,計數無比例分母；須保留未歸里及來源涵蓋狀態,操作口徑已確認，最終年度里層級計數尚不可建立,本次僅逐年候選N；不等於所有交通事故或暴露校正事故率
1,total_injuries,受傷人數,sum(有效事故受傷人數)，全無有效值則NA,同記有效N／缺失N／結構異常N；缺失不補0，不稱已觀測和為完整總數,總量欄候選；未來按里年彙整,不以第一當事人受傷程度代替；與A多人受傷相關，不是獨立驗證
2,total_fatalities_24h,死亡人數,暫定sum(有效死亡人數)，全無有效值則NA,非負整數結構檢查；記有效N、缺失與異常N,暫定英文名；PDF p.1支持24小時，但CSV發布映射尚缺,未認證最終死亡統計口徑，不與2–30日死亡直接相加
3,total_fatalities_2_30d,2-30日死亡人數,暫定sum(有效原欄人數)，全無有效值則NA,記有效N、缺失N與異常N；保留原始欄意,欄名與PDF個人代碼支持候選，事故追蹤彙整規格未取得,個人2–30日程度碼不能代替事件總量，亦未認證可與另一死亡欄無重複相加
4,fatal_accident_ratio,死亡人數（24小時候選口徑）,有效死亡人數≥1的事件數 / 死亡人數已知有效事件數,只用該死亡欄的有效分母；分母0為NA,暫定24小時版本，須學生確認且補發布映射,不是死亡人數除以事故數；不合併2–30日，不從第一當事人傷亡程度推算
5,injuries_per_crash,受傷人數,有效事故受傷人數總和 / 有效受傷人數事件N,用有效N而非含缺失的全部候選N；分母0為NA,可作條件於傷亡資料已知的平均；未建里年數值,有缺失時不能當全體事故平均，需同報涵蓋率；與A傷亡特徵非完全獨立


**表 22：傷亡原欄的有效N、缺失及已觀測和；三欄保持分列**

,year,原欄,全部候選N,有效N,原始缺失N,非缺失結構異常N,已觀測有效人數和,資料完整總數可否認證,正值事件數,每有效事件原欄平均
0,2024,受傷人數,22368,22367,1,0,29774.0,否，仍有缺失或發布口徑待補,22314,1.331158
1,2024,死亡人數,22368,22359,9,0,70.0,否，仍有缺失或發布口徑待補,68,0.003131
2,2024,2-30日死亡人數,22368,22359,9,0,4.0,否，仍有缺失或發布口徑待補,4,0.000179
3,2025,受傷人數,22762,22761,1,0,30201.0,否，仍有缺失或發布口徑待補,22716,1.326875
4,2025,死亡人數,22762,22755,7,0,58.0,否，仍有缺失或發布口徑待補,57,0.002549
5,2025,2-30日死亡人數,22762,22752,10,0,3.0,否，仍有缺失或發布口徑待補,3,0.000132


{'year': 2024, '死亡人數正值事件': 68, '第一當事人程度碼1': 29, '2至30日原欄正值事件': 4, '第一當事人程度碼5': 9, 'interpretation': 'different units; no substitution, no automatic reconciliation or mortality-field combination'}
{'year': 2025, '死亡人數正值事件': 57, '第一當事人程度碼1': 28, '2至30日原欄正值事件': 3, '第一當事人程度碼5': 5, 'interpretation': 'different units; no substitution, no automatic reconciliation or mortality-field combination'}


**表 22B：六項C候選的逐年全市記憶體診斷；非正式年度資料集**

,year,候選變數,逐年全市診斷值,有效事件N,來源缺失N,狀態
0,2024,annual_accident_count,22368.000000,22368,0,未GIS，非最終里年事故數
1,2024,total_injuries,29774.000000,22367,1,已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定
2,2024,total_fatalities_24h,70.000000,22359,9,已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定
3,2024,total_fatalities_2_30d,4.000000,22359,9,已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定
4,2024,fatal_accident_ratio,0.003041,22359,9,僅死亡人數原欄正值事件比例；24小時映射暫定，不合併2–30日
5,2024,injuries_per_crash,1.331158,22367,1,條件於受傷人數已知的事件平均，非全體事故完整平均
6,2025,annual_accident_count,22762.000000,22762,0,未GIS，非最終里年事故數
7,2025,total_injuries,30201.000000,22761,1,已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定
8,2025,total_fatalities_24h,58.000000,22755,7,已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定
9,2025,total_fatalities_2_30d,3.000000,22752,10,已觀測有效人數和；非缺失補零後的完整總數，死亡口徑仍暫定


**表 22C：個人程度碼與事件總量的單向條件式一致性；不要求兩種單位數值相等**

,year,第一當事人程度碼,對應事故總量原欄,程度碼符合N,事故總量已知正值,事故總量為0的待查N,事故總量缺失N,判讀
0,2024,1,死亡人數,29,29,0,0,若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補
1,2024,5,2-30日死亡人數,9,3,6,0,若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補
2,2024,2,受傷人數,10065,10065,0,0,若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補
3,2025,1,死亡人數,28,28,0,0,若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補
4,2025,5,2-30日死亡人數,5,1,4,0,若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補
5,2025,2,受傷人數,10281,10281,0,0,若同事件同追蹤時點口徑，第一當事人有此傷亡應支持總量正值；目前發布映射／更新時點未證實，先標記不修補


## 8. Cleaning Rule Proposals and Sample Impact
R01 是已確認的候選母體選取；其餘按狀態區分已核對定義與待審候選處理。
以下只建立布林標記、計數與候選分母，不覆寫任何原欄，不輸出清理表。
不同規則可重疊，受影響筆數不能相加當作總排除人數。

表25傷亡欄的「有效」僅指已知、可解析、有限非負整數，不等同CSV發布語義或R20條件式一致性已通過。上述6／4筆待查紀錄仍包含在結構有效N中；本次未擅自將其改為缺失或排除，後續若調整C分母須另經審查。


In [10]:
RULE_DEFINITIONS = [{'規則ID': 'R01', '適用欄': '當事人序號', '原始問題': '兩年原始當事人口徑不同', '建議處理': '依原始序號選1；其他當事人保留在raw', '規則狀態': '已確認：第一當事人候選口徑', '整筆事故資格': '是，決定候選事件母體', '局部分母影響': '否', '依據及待確認': '其他當事人不等同重複事故；無官方ID', '判定條件': '原始當事人序號 != 1；計數以未篩選raw為母體'}, {'規則ID': 'R02', '適用欄': '完整列／候選事件鍵', '原始問題': '可能重複或同時同地事件', '建議處理': '重複時標記並停止確認；不按候選鍵自動去重', '規則狀態': '已確認：鍵僅診斷', '整筆事故資格': '否，不自動刪除', '局部分母影響': '否', '依據及待確認': '真實事件識別及重複衝突規則仍需來源', '判定條件': '候選完整列重複或KEY重複（keep=False；KEY含缺失）'}, {'規則ID': 'R03', '適用欄': '年月日', '原始問題': '無法解析日期或來源年度不符', '建議處理': '標記；週末等日期特徵保留NA，不牽連其他變數', '規則狀態': '建議待審：特徵局部有效分母', '整筆事故資格': '否', '局部分母影響': '是，日期特徵', '依據及待確認': '若來源年本身有衝突另查，不以檔名硬覆寫', '判定條件': '日期不可解析／年月日非整數／民國年 != source_year-1911'}, {'規則ID': 'R04', '適用欄': '時／分', '原始問題': '非整數或超出時分範圍', '建議處理': '標記；僅退出時段分母，不將未知時分當非尖峰', '規則狀態': '已確認操作端點；局部規則待審', '整筆事故資格': '否', '局部分母影響': '是，三個時段特徵', '依據及待確認': '24:00等需發布規格，不自動歸次日', '判定條件': '時分非有限非負整數，或h不在0–23、m不在0–59'}, {'規則ID': 'R05', '適用欄': '事故類型及型態', '原始問題': '缺失或不符可核對碼集', '建議處理': '保留事故；型態特徵為NA，使用各自有效分母', '規則狀態': '建議待審；碼值已核對', '整筆事故資格': '否', '局部分母影響': '是，追撞／側撞／人與車', '依據及待確認': '其他有效碼仍是分母，不把其他當缺失', '判定條件': '事故型態缺失或不屬PDF可核對印列碼1–38'}, {'規則ID': 'R06', '適用欄': '事故位置', '原始問題': '缺失或不符可核對碼集', '建議處理': '保留事故；只影響intersection分母', '規則狀態': '建議待審；碼值已核對', '整筆事故資格': '否', '局部分母影響': '是，路口特徵', '依據及待確認': '嚴格路口內只改分子，不誤改分母', '判定條件': '事故位置缺失或不屬{1,…,20,22,23,24,25}'}, {'規則ID': 'R07', '適用欄': '受傷人數', '原始問題': '缺失／非負整數結構異常', '建議處理': '標記；不補0；多人受傷與C傷亡用有效N', '規則狀態': '已確認缺失非0；結構規則候選', '整筆事故資格': '否', '局部分母影響': '是，multi_injury與C', '依據及待確認': '不以個人受傷程度替代，不擅設人數上限', '判定條件': '受傷人數缺失、不可解析、非有限、負值或非整數'}, {'規則ID': 'R08', '適用欄': '死亡人數', '原始問題': '缺失／結構異常及口徑待確認', '建議處理': '保留原欄，總和與有效／缺失N同報', '規則狀態': '建議待審；24小時名稱暫定', '整筆事故資格': '否', '局部分母影響': '是，對應C', '依據及待確認': '需CSV發布映射，不能直接與2–30日加總', '判定條件': '死亡人數缺失、不可解析、非有限、負值或非整數'}, {'規則ID': 'R09', '適用欄': '2-30日死亡人數', '原始問題': '缺失／結構異常及追蹤彙整未知', '建議處理': '獨立保留原欄，缺失非0，不與另一死亡欄合併', '規則狀態': '建議待審', '整筆事故資格': '否', '局部分母影響': '是，對應C', '依據及待確認': '跟蹤／涵蓋及重複計算規格待補', '判定條件': '2–30日死亡人數缺失、不可解析、非有限、負值或非整數'}, {'規則ID': 'R10', '適用欄': 'X／Y', '原始問題': '缺失、非數值或非有限值', '建議處理': '標記coordinate_unavailable，保留市級診斷；未來GIS資格另審', '規則狀態': '候選：空間可用性標記', '整筆事故資格': '本單元否；未來歸屬可能受限', '局部分母影響': '只影響空間連接，不影響已知其他特徵', '依據及待確認': '即使可解析也未認證CRS；不直接丟棄', '判定條件': '至少一座標文字空白、不能解析或非有限數值'}, {'規則ID': 'R11', '適用欄': '速限', '原始問題': '0的意義未知', '建議處理': '保留原值並標記；試算排除於速限補充數值分母', '規則狀態': '建議待學生確認，不把0視必然錯誤', '整筆事故資格': '否', '局部分母影響': '是，若選用速限補充', '依據及待確認': '不沿用舊夜間研究整筆排除0的規則', '判定條件': '原始速限數值恰為0；不推定為無速限或錯誤'}, {'規則ID': 'R12', '適用欄': '速限', '原始問題': '502極端值', '建議處理': '保留及標記，待查；試算速限分母排除此值', '規則狀態': '建議待學生確認', '整筆事故資格': '否', '局部分母影響': '是，速限補充敏感度', '依據及待確認': '不自動改成50，亦不以任意速限上限宣稱非法', '判定條件': '原始速限數值恰為502；不推定應為50'}, {'規則ID': 'R13', '適用欄': '第一當事人年齡', '原始問題': '負值含-1定義未知', '建議處理': '標記；候選年齡數值分母不納負數，事故仍保留', '規則狀態': '建議待學生確認', '整筆事故資格': '否', '局部分母影響': '是，年齡補充', '依據及待確認': '先查特殊碼，不改原始年齡', '判定條件': '原始年齡數值 < 0；缺失另列'}, {'規則ID': 'R14', '適用欄': '第一當事人年齡', '原始問題': '0可能是真實嬰幼兒或未知碼', '建議處理': '保留原值、只作標記；另示若視未知的分母影響', '規則狀態': '未確認，不能一律判錯', '整筆事故資格': '否', '局部分母影響': '僅候選年齡敏感度', '依據及待確認': '需聯合身分類別及官方規格，不能只看0', '判定條件': '原始年齡數值 == 0；缺失另列'}, {'規則ID': 'R15', '適用欄': '第一當事人年齡', '原始問題': '高於100不必然錯誤', '建議處理': '保留且標記；只示年齡1–100候選敏感度', '規則狀態': '未確認上限，不直接截尾', '整筆事故資格': '否', '局部分母影響': '僅候選年齡敏感度', '依據及待確認': '真實高齡與未知填碼需來源證據', '判定條件': '原始年齡數值 > 100；僅標記，非生物學或官方上限'}, {'規則ID': 'R16', '適用欄': '第一當事人性別', '原始問題': '3–5及缺失不是一般男／女', '建議處理': '保留原碼；若描述男女比例，分母只取1、2', '規則狀態': '建議待審，特殊碼已確認', '整筆事故資格': '否', '局部分母影響': '是，男女補充比例', '依據及待確認': '3–5是表單合法狀態，不代表女性或缺失', '判定條件': '若描述男／女：性別不在{1,2}，含特殊3–5及缺失'}, {'規則ID': 'R17', '適用欄': '第一當事人車種', '原始問題': '缺失或碼外', '建議處理': '保留事故，只影響第一當事人運具補充', '規則狀態': '建議待審', '整筆事故資格': '否', '局部分母影響': '是，車種補充', '依據及待確認': '不能拿第一當事人代表全部涉入者', '判定條件': '車種缺失或不在PDF p.3㉖所列字串碼'}, {'規則ID': 'R18', '適用欄': '道路照明設備／光線', '原始問題': '跨年語義等價未證實', '建議處理': '保留原欄及年份；不正式合併、重編碼或納主A', '規則狀態': '明確保留未知；再用待學生確認', '整筆事故資格': '否', '局部分母影響': '不影響目前九項A', '依據及待確認': '受影響N指語義待證範圍，不是排除／錯誤N', '判定條件': '兩年全部候選列均受跨年照明語意待證限制；不是無效列'}, {'規則ID': 'R19', '適用欄': '全部欄位', '原始問題': '任一欄缺失就刪整筆會失去有效資訊', '建議處理': '拒絕全欄complete-case；只試算其代價，採各變數分母', '規則狀態': '已確認：不採全欄刪除', '整筆事故資格': '否；本規則禁止整筆刪除', '局部分母影響': '各特徵獨立', '依據及待確認': '2024主要肇因全缺失，全欄完整案例會歸零', '判定條件': '任一原始欄位isna；僅試算全欄complete-case錯誤政策'}, {'規則ID': 'R20', '適用欄': '第一當事人受傷程度×事故傷亡總量', '原始問題': '個人程度碼5對應2–30日事件總量為0，兩欄追蹤／更新口徑未證實', '判定條件': '程度碼1且死亡人數=0，或碼5且2–30日死亡人數=0，或碼2且受傷人數=0；缺失另列', '建議處理': '新增conditional_casualty_inconsistency標記，保留兩欄原值，查證追蹤／更新規格；不以個人碼填補總量', '規則狀態': '已觀測疑點；處理方式待學生及來源確認', '整筆事故資格': '否，不自動刪事故或重設其他特徵', '局部分母影響': '目前只標記；若將來限制C，須另審查且與缺失N分列', '依據及待確認': 'PDF p.1／p.3；同事件同追蹤時點是條件，現階段不推定哪個欄位錯誤'}]
impact=[];policy_examples=[]
for year,frame in candidates.items():
    v=VALID[year];age=NUM[year]['年齡'];speed=NUM[year]['速限-速度限制']
    affected={
      'R02':frame.duplicated(KEY,keep=False)|frame.duplicated(keep=False),
      'R03':~v['date'],'R04':~v['time'],'R05':~v['type'],'R06':~v['position'],
      'R07':~v['injuries'],'R08':~v['fatalities_24h'],'R09':~v['fatalities_2_30d'],
      'R10':~v['coordinates_parseable'],'R11':speed.eq(0),'R12':speed.eq(502),
      'R13':age.lt(0),'R14':age.eq(0),'R15':age.gt(100),
      'R16':~v['sex_male_female'],'R17':~v['vehicle'],
      'R18':pd.Series(True,index=frame.index),'R19':frame.isna().any(axis=1),
    }
    affected['R20']=((frame['受傷程度'].eq(1)&NUM[year]['死亡人數'].eq(0))
                      | (frame['受傷程度'].eq(5)&NUM[year]['2-30日死亡人數'].eq(0))
                      | (frame['受傷程度'].eq(2)&NUM[year]['受傷人數'].eq(0)))
    for definition in RULE_DEFINITIONS:
        rid=definition['規則ID']
        count=int(raw[year]['當事人序號'].ne(1).sum()) if rid=='R01' else int(affected[rid].fillna(False).sum())
        impact.append({'規則ID':rid,'year':year,'受影響筆數':count,
                       '計數母體':'原始當事人列' if rid=='R01' else '第一當事人候選列',
                       '是否已正式清理':'否；R01僅建立記憶體候選選取'})
    speed_candidate=nonnegative_integer(frame['速限-速度限制'])&speed.gt(0)&speed.ne(502)
    age_base=nonnegative_integer(frame['年齡'])
    for policy,valid_mask in [
      ('保留候選事件母體',pd.Series(True,index=frame.index)),
      ('只在速限補充分母排除缺失／0／502（候選）',speed_candidate),
      ('年齡非負整數；保留0與高齡（候選）',age_base),
      ('年齡1–100敏感度；非已確認排除規則',age_base&age.between(1,100)),
      ('錯誤政策示例：要求全原始欄無缺失（不採用）',frame.notna().all(axis=1)),
    ]:
        policy_examples.append({'year':year,'記憶體情境':policy,'情境可用N':int(valid_mask.sum()),
                               '候選N':len(frame),'與母體差額':int((~valid_mask).sum()),'是否實際刪除':'否'})
rule_impacts=pd.DataFrame(impact)
rule_summary=pd.DataFrame(RULE_DEFINITIONS).merge(
    rule_impacts.pivot(index='規則ID',columns='year',values='受影響筆數').rename(columns={2024:'2024受影響',2025:'2025受影響'}),
    on='規則ID',validate='one_to_one')
for columns in [['規則ID','適用欄','原始問題','判定條件','建議處理','規則狀態'],
                ['規則ID','整筆事故資格','局部分母影響','2024受影響','2025受影響','依據及待確認']]:
    show('表 23：正式候選清理規則及實際影響筆數',rule_summary[columns],block=10)
show('表 24：互不等價的候選分母情境；不會刪除記憶體以外資料',pd.DataFrame(policy_examples),block=10)

flow=[]
for year,frame in candidates.items():
    flow.append({'year':year,'階段／獨立條件':'原始當事人紀錄','N':len(raw[year]),'涵蓋率分母':'原始當事人列',
                 '分母N':len(raw[year]),'涵蓋率%':100.,'非巢狀說明':'來源列數，不等於事故數'})
    flow.append({'year':year,'階段／獨立條件':'第一當事人候選事件','N':len(frame),'涵蓋率分母':'原始當事人列',
                 '分母N':len(raw[year]),'涵蓋率%':100*len(frame)/len(raw[year]),'非巢狀說明':'唯一已確認的口徑選取；未GIS配對'})
    for label,key,note in [
      ('具有效日期時間','datetime','曆法、來源年、時分皆有效'),
      ('具有效事故型態','type','PDF印列碼；與日期／傷亡／座標條件獨立'),
      ('具有效事故位置','position','只供位置特徵有效分母'),
      ('具有效受傷人數','injuries','人數非負整數；不要求兩個死亡欄已知'),
      ('具有效死亡人數原欄','fatalities_24h','24小時映射仍暫定；不要求受傷人數完整'),
      ('具有效2–30日死亡原欄','fatalities_2_30d','與另一死亡欄分開，不合併統計口徑'),
      ('三個傷亡人數欄皆有效','casualties_all_known','交集只作特定C完整性診斷，不是通用納樣門檻'),
      ('具可解析座標','coordinates_parseable','只代表可解析；CRS與里內歸屬未認證'),
    ]:
        n=int(VALID[year][key].sum())
        flow.append({'year':year,'階段／獨立條件':label,'N':n,'涵蓋率分母':'第一當事人候選事件',
                     '分母N':len(frame),'涵蓋率%':100*n/len(frame),'非巢狀說明':note})
sample_flow=pd.DataFrame(flow)
show('表 25：Sample Flow Table；候選後各條件獨立，不是逐步縮減漏斗',sample_flow,block=11)
print('請勿將表25各列差額加總或畫成依序刪除的樣本流程。')


**表 23：正式候選清理規則及實際影響筆數**

,規則ID,適用欄,原始問題,判定條件,建議處理,規則狀態
0,R01,當事人序號,兩年原始當事人口徑不同,原始當事人序號 != 1；計數以未篩選raw為母體,依原始序號選1；其他當事人保留在raw,已確認：第一當事人候選口徑
1,R02,完整列／候選事件鍵,可能重複或同時同地事件,候選完整列重複或KEY重複（keep=False；KEY含缺失）,重複時標記並停止確認；不按候選鍵自動去重,已確認：鍵僅診斷
2,R03,年月日,無法解析日期或來源年度不符,日期不可解析／年月日非整數／民國年 != source_year-1911,標記；週末等日期特徵保留NA，不牽連其他變數,建議待審：特徵局部有效分母
3,R04,時／分,非整數或超出時分範圍,時分非有限非負整數，或h不在0–23、m不在0–59,標記；僅退出時段分母，不將未知時分當非尖峰,已確認操作端點；局部規則待審
4,R05,事故類型及型態,缺失或不符可核對碼集,事故型態缺失或不屬PDF可核對印列碼1–38,保留事故；型態特徵為NA，使用各自有效分母,建議待審；碼值已核對
5,R06,事故位置,缺失或不符可核對碼集,"事故位置缺失或不屬{1,…,20,22,23,24,25}",保留事故；只影響intersection分母,建議待審；碼值已核對
6,R07,受傷人數,缺失／非負整數結構異常,受傷人數缺失、不可解析、非有限、負值或非整數,標記；不補0；多人受傷與C傷亡用有效N,已確認缺失非0；結構規則候選
7,R08,死亡人數,缺失／結構異常及口徑待確認,死亡人數缺失、不可解析、非有限、負值或非整數,保留原欄，總和與有效／缺失N同報,建議待審；24小時名稱暫定
8,R09,2-30日死亡人數,缺失／結構異常及追蹤彙整未知,2–30日死亡人數缺失、不可解析、非有限、負值或非整數,獨立保留原欄，缺失非0，不與另一死亡欄合併,建議待審
9,R10,X／Y,缺失、非數值或非有限值,至少一座標文字空白、不能解析或非有限數值,標記coordinate_unavailable，保留市級診斷；未來GIS資格另審,候選：空間可用性標記


,規則ID,適用欄,原始問題,判定條件,建議處理,規則狀態
10,R11,速限,0的意義未知,原始速限數值恰為0；不推定為無速限或錯誤,保留原值並標記；試算排除於速限補充數值分母,建議待學生確認，不把0視必然錯誤
11,R12,速限,502極端值,原始速限數值恰為502；不推定應為50,保留及標記，待查；試算速限分母排除此值,建議待學生確認
12,R13,第一當事人年齡,負值含-1定義未知,原始年齡數值 < 0；缺失另列,標記；候選年齡數值分母不納負數，事故仍保留,建議待學生確認
13,R14,第一當事人年齡,0可能是真實嬰幼兒或未知碼,原始年齡數值 == 0；缺失另列,保留原值、只作標記；另示若視未知的分母影響,未確認，不能一律判錯
14,R15,第一當事人年齡,高於100不必然錯誤,原始年齡數值 > 100；僅標記，非生物學或官方上限,保留且標記；只示年齡1–100候選敏感度,未確認上限，不直接截尾
15,R16,第一當事人性別,3–5及缺失不是一般男／女,"若描述男／女：性別不在{1,2}，含特殊3–5及缺失",保留原碼；若描述男女比例，分母只取1、2,建議待審，特殊碼已確認
16,R17,第一當事人車種,缺失或碼外,車種缺失或不在PDF p.3㉖所列字串碼,保留事故，只影響第一當事人運具補充,建議待審
17,R18,道路照明設備／光線,跨年語義等價未證實,兩年全部候選列均受跨年照明語意待證限制；不是無效列,保留原欄及年份；不正式合併、重編碼或納主A,明確保留未知；再用待學生確認
18,R19,全部欄位,任一欄缺失就刪整筆會失去有效資訊,任一原始欄位isna；僅試算全欄complete-case錯誤政策,拒絕全欄complete-case；只試算其代價，採各變數分母,已確認：不採全欄刪除
19,R20,第一當事人受傷程度×事故傷亡總量,個人程度碼5對應2–30日事件總量為0，兩欄追蹤／更新口徑未證實,程度碼1且死亡人數=0，或碼5且2–30日死亡人數=0，或碼2且受傷人數=0；缺失另列,新增conditional_casualty_inconsistency標記，保留兩欄原值，查證追蹤／更新規格；不以個人碼填補總量,已觀測疑點；處理方式待學生及來源確認


**表 23：正式候選清理規則及實際影響筆數**

,規則ID,整筆事故資格,局部分母影響,2024受影響,2025受影響,依據及待確認
0,R01,是，決定候選事件母體,否,29442,0,其他當事人不等同重複事故；無官方ID
1,R02,否，不自動刪除,否,0,0,真實事件識別及重複衝突規則仍需來源
2,R03,否,是，日期特徵,0,0,若來源年本身有衝突另查，不以檔名硬覆寫
3,R04,否,是，三個時段特徵,0,0,24:00等需發布規格，不自動歸次日
4,R05,否,是，追撞／側撞／人與車,9,12,其他有效碼仍是分母，不把其他當缺失
5,R06,否,是，路口特徵,9,12,嚴格路口內只改分子，不誤改分母
6,R07,否,是，multi_injury與C,1,1,不以個人受傷程度替代，不擅設人數上限
7,R08,否,是，對應C,9,7,需CSV發布映射，不能直接與2–30日加總
8,R09,否,是，對應C,9,10,跟蹤／涵蓋及重複計算規格待補
9,R10,本單元否；未來歸屬可能受限,只影響空間連接，不影響已知其他特徵,1,2,即使可解析也未認證CRS；不直接丟棄


,規則ID,整筆事故資格,局部分母影響,2024受影響,2025受影響,依據及待確認
10,R11,否,是，若選用速限補充,328,315,不沿用舊夜間研究整筆排除0的規則
11,R12,否,是，速限補充敏感度,1,0,不自動改成50，亦不以任意速限上限宣稱非法
12,R13,否,是，年齡補充,1,0,先查特殊碼，不改原始年齡
13,R14,否,僅候選年齡敏感度,222,217,需聯合身分類別及官方規格，不能只看0
14,R15,否,僅候選年齡敏感度,11,18,真實高齡與未知填碼需來源證據
15,R16,否,是，男女補充比例,292,260,3–5是表單合法狀態，不代表女性或缺失
16,R17,否,是，車種補充,91,157,不能拿第一當事人代表全部涉入者
17,R18,否,不影響目前九項A,22368,22762,受影響N指語義待證範圍，不是排除／錯誤N
18,R19,否；本規則禁止整筆刪除,各特徵獨立,22368,255,2024主要肇因全缺失，全欄完整案例會歸零
19,R20,否，不自動刪事故或重設其他特徵,目前只標記；若將來限制C，須另審查且與缺失N分列,6,4,PDF p.1／p.3；同事件同追蹤時點是條件，現階段不推定哪個欄位錯誤


... (2 outputs left out)

**表 25：Sample Flow Table；候選後各條件獨立，不是逐步縮減漏斗**

,year,階段／獨立條件,N,涵蓋率分母,分母N,涵蓋率%,非巢狀說明
0,2024,原始當事人紀錄,51810,原始當事人列,51810,100.000000,來源列數，不等於事故數
1,2024,第一當事人候選事件,22368,原始當事人列,51810,43.173133,唯一已確認的口徑選取；未GIS配對
2,2024,具有效日期時間,22368,第一當事人候選事件,22368,100.000000,曆法、來源年、時分皆有效
3,2024,具有效事故型態,22359,第一當事人候選事件,22368,99.959764,PDF印列碼；與日期／傷亡／座標條件獨立
4,2024,具有效事故位置,22359,第一當事人候選事件,22368,99.959764,只供位置特徵有效分母
5,2024,具有效受傷人數,22367,第一當事人候選事件,22368,99.995529,人數非負整數；不要求兩個死亡欄已知
6,2024,具有效死亡人數原欄,22359,第一當事人候選事件,22368,99.959764,24小時映射仍暫定；不要求受傷人數完整
7,2024,具有效2–30日死亡原欄,22359,第一當事人候選事件,22368,99.959764,與另一死亡欄分開，不合併統計口徑
8,2024,三個傷亡人數欄皆有效,22359,第一當事人候選事件,22368,99.959764,交集只作特定C完整性診斷，不是通用納樣門檻
9,2024,具可解析座標,22367,第一當事人候選事件,22368,99.995529,只代表可解析；CRS與里內歸屬未認證


,year,階段／獨立條件,N,涵蓋率分母,分母N,涵蓋率%,非巢狀說明
11,2025,第一當事人候選事件,22762,原始當事人列,22762,100.000000,唯一已確認的口徑選取；未GIS配對
12,2025,具有效日期時間,22762,第一當事人候選事件,22762,100.000000,曆法、來源年、時分皆有效
13,2025,具有效事故型態,22750,第一當事人候選事件,22762,99.947281,PDF印列碼；與日期／傷亡／座標條件獨立
14,2025,具有效事故位置,22750,第一當事人候選事件,22762,99.947281,只供位置特徵有效分母
15,2025,具有效受傷人數,22761,第一當事人候選事件,22762,99.995607,人數非負整數；不要求兩個死亡欄已知
16,2025,具有效死亡人數原欄,22755,第一當事人候選事件,22762,99.969247,24小時映射仍暫定；不要求受傷人數完整
17,2025,具有效2–30日死亡原欄,22752,第一當事人候選事件,22762,99.956067,與另一死亡欄分開，不合併統計口徑
18,2025,三個傷亡人數欄皆有效,22752,第一當事人候選事件,22762,99.956067,交集只作特定C完整性診斷，不是通用納樣門檻
19,2025,具可解析座標,22760,第一當事人候選事件,22762,99.991213,只代表可解析；CRS與里內歸屬未認證


請勿將表25各列差額加總或畫成依序刪除的樣本流程。


## 9. Confirmed Findings, Candidate Decisions and Handoff

**已確認的範圍：** 依固定版本重新核對來源、第一當事人操作口徑、候選鍵診斷、
PDF主要型態／位置代碼、各欄缺失與特殊值、每項特徵的實際可判定分母。
這不是官方事件ID或完成GIS樣本的認證，也沒有產出清理後事故表。

**待學生審查的候選規則：** 速限0／502在補充特徵的處理、年齡0／負值／極高值及自然人分母、
照明欄位等價證據與是否使用、fatal_accident_ratio的死亡時間窗、
是否將multi_injury留在A或僅放C，以及低樣本／比例平滑／最終特徵等後續設定。
本次記憶體試算不等於接受任何候選刪除或補值規則。

**待外部來源確認：** 官方事件及2025第一當事人發布口徑、CSV死亡欄／追蹤彙整規格、
照明與部分碼表版本、座標基準。碼表不完整不阻止其他已定義特徵的診斷；
CRS未認證會阻止正式空間歸里，但不阻止下一個獲授權的唯讀驗證單元。

**重現方式：** 在工作包內以 `.venv/Scripts/python.exe` 啟動Jupyter，選DSSI Urban Crash，
從乾淨kernel執行全部code cells。Notebook本身不連網、不寫CSV／processed、不呼叫其他Notebook；
原始資料或上游文件hash不同時先停止查證，不改預期hash強制通過。
所有來源頁面查閱日固定記為2026-11-26；之後重跑不代表重新查閱官方頁。

完成Unit 1.1後停止等待學生審查，不自動進入Unit 1.2。人口、里界及都市計畫仍由後續單元處理。

另外須確認R20傷亡條件式不一致標記的後續處理；目前不刪除6／4筆疑點，不改任何傷亡人數，九項A的原始有效分母不因本標記改變。


In [11]:
for relative,expected in EXPECTED_SHA256.items():
    assert sha256_file(project_file(relative))==expected,relative
assert OUTPUTS=={}
assert len(schema_table)==49
assert len(feature_coverage_table)==18
assert len(rule_summary)==20
print({'all_bound_inputs_unchanged':True,'upstream_raw_document_unchanged':True,
       'input_files_checked':len(EXPECTED_SHA256),'schema_rows':len(schema_table),
       'feature_year_coverage_rows':len(feature_coverage_table),'cleaning_rules':len(rule_summary),
       'new_processed_datasets':0,'cross_year_accident_row_merge':False,'GIS_join':False,
       'CRS_assigned':False,'formal_cleaning_applied':False,'model_or_inference_run':False})


{'all_bound_inputs_unchanged': True, 'upstream_raw_document_unchanged': True, 'input_files_checked': 4, 'schema_rows': 49, 'feature_year_coverage_rows': 18, 'cleaning_rules': 20, 'new_processed_datasets': 0, 'cross_year_accident_row_merge': False, 'GIS_join': False, 'CRS_assigned': False, 'formal_cleaning_applied': False, 'model_or_inference_run': False}
